# Madrid bus electrification: line, depot and city model
**IE Sustainability Datathon (Kearney) · Deliverable 1 · Main notebook**

This notebook produces the four mandatory outputs `File_1.csv`–`File_4.csv`, the BI visualisation and every check the jury needs. It is built in dependency order: baseline → lines (File 2) → depots (File 3) → prioritisation → scenarios (File 4) → benchmark (File 1) → visualisation → compliance.

**How to read it.** Each section follows the same pattern: a text cell (what, why, sources, V/D/P/A label), a short code cell, then a visible output — a table, a chart or a `CHECK … PASS` line. The jury reads saved outputs only; a step that prints nothing counts as not done.

| Section | Output | Owner |
|---|---|---|
| 0 Setup and parameters | parameter table | [owner] |
| 1 Data acquisition | data register | [owner] |
| 2 Reference day | chosen dates | [owner] |
| 3 Fleet baseline 2025 | baseline, S0 pace | [owner] |
| 4 Line model | File 2 metrics | [owner] |
| 5 Depot allocation | allocation + reconciliation | [owner] |
| 6 Depot power and readiness | File 3 | [owner] |
| 7 Prioritisation | completes File 2 | [owner] |
| 8 Scenarios | File 4 | [owner] |
| 9 Benchmark | File 1 | [owner] |
| 10 Visualisation | HTML + static views | [owner] |
| 11 Assumptions register | register | [owner] |
| 12 Final compliance | read-back + checks | [owner] |

## 0 · Setup and fixed parameters
Libraries, folders and the brief's fixed parameters (section 4.5). Every parameter lives in `P`; no other cell hard-codes a number. Provenance labels: **V** verified · **D** derived · **P** proxy · **A** assumption.

In [ ]:
# Environment: runs as-is in Google Colab (Python 3.10+).
%pip install -q pandas numpy pyproj shapely folium plotly kaleido requests scikit-learn

import os, io, json, zipfile, datetime as dt, hashlib, textwrap
import numpy as np, pandas as pd, requests

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

BASE = "/content/datathon" if os.path.exists("/content") else os.path.abspath("..")
RAW, DERIVED, OUT = (os.path.join(BASE, d) for d in ("data_raw", "data_derived", "outputs"))
for d in (RAW, DERIVED, OUT):
    os.makedirs(d, exist_ok=True)

RUN_DATE = dt.date.today().isoformat()
import sys, platform
print(f"Run date {RUN_DATE} | Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}")
print("Folders:", RAW, DERIVED, OUT, sep="\n  ")

**Helpers.** `fetch()` downloads each source once and logs URL, date, size and checksum; `assume()` labels every value V, D, P or A for the assumptions register; `check()` prints a visible PASS / FAIL line.

In [ ]:
# ---- Data register: every downloaded or manually obtained source is logged here.
DATA_REGISTER = []

def fetch(url, filename, note="", manual=False):
    """Download a public file into data_raw/ and log source, date, size and checksum.
    manual=True: the file was obtained by hand (registration/API); it must already be in data_raw/."""
    path = os.path.join(RAW, filename)
    if not manual and not os.path.exists(path):
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        with open(path, "wb") as f:
            f.write(r.content)
    size = os.path.getsize(path) if os.path.exists(path) else None
    sha = hashlib.sha256(open(path, "rb").read()).hexdigest()[:12] if size else None
    DATA_REGISTER.append(dict(file=filename, url=url, downloaded=RUN_DATE, bytes=size,
                              sha256_12=sha, obtained="manual" if manual else "script", note=note))
    return path

def show_register():
    display(pd.DataFrame(DATA_REGISTER))

# ---- Provenance labels (brief section 4.6): V verified, D derived, P proxy, A assumption.
ASSUMPTIONS = []

def assume(variable, value, label, evidence, impact="", sensitivity=""):
    assert label in {"V", "D", "P", "A"}, "label must be V, D, P or A"
    ASSUMPTIONS.append(dict(variable=variable, value=value, label=label, evidence=evidence,
                            impact=impact, sensitivity=sensitivity))
    return value

# ---- Visible checks: every step prints CHECK ... PASS / FAIL.
CHECKS = []

def check(name, ok, detail=""):
    CHECKS.append(dict(check=name, result="PASS" if ok else "FAIL", detail=detail))
    print(f"CHECK {'PASS' if ok else 'FAIL'} | {name}" + (f" | {detail}" if detail else ""))
    return ok

**Fixed parameters (brief §4.5).** Every mandatory value lives in `P`; no other cell hard-codes a number, so sensitivities only change inputs.

In [ ]:
# Fixed parameters from the official brief, section 4.5. Do not hard-code these numbers anywhere else.
P = dict(
    kwh_km      = {"Standard": 1.30, "Articulated": 1.85},   # battery energy incl. HVAC
    range_km    = {"Standard": 300,  "Articulated": 240},    # effective daily range (75% of tender minimum)
    cng_kg_km   = {"Standard": 0.45, "Articulated": 0.65},
    charger_kw  = 100,
    window_h    = 7,                                          # 23:00-06:00
    charge_eff  = 0.90,
    bus_price_eur = {"Standard": 615_000, "Articulated": 800_000},  # excl. VAT
    benchmark_kwh_km = 1.30,
    service_days = 365,
    # TODO(owner: emissions) confirm value, unit and boundary against the cited page before submission
    cng_kgco2_per_kg = assume("CO2 factor CNG", 2.7175, "V", "MITECO emission factors (inventory annex VII)", "Files 2, 4"),
    grid_gco2_kwh    = assume("CO2 factor electricity Spain 2025", 108.4, "V", "Red Electrica 2025 generation emissions", "Files 1, 2, 4"),
)
display(pd.DataFrame([(k, v) for k, v in P.items()], columns=["parameter", "value"]))

**GTFS helpers.** Read a feed, find the services active on a date (`calendar`, then `calendar_dates` exceptions), and measure trip length from `shape_dist_traveled` or the shape geometry.

In [ ]:
# ---- Generic GTFS helpers (shared logic with the benchmark notebook).
def read_gtfs(zip_path, tables=("agency","routes","trips","stop_times","calendar","calendar_dates","shapes","feed_info")):
    feed = {}
    with zipfile.ZipFile(zip_path) as z:
        names = set(z.namelist())
        for t in tables:
            if f"{t}.txt" in names:
                feed[t] = pd.read_csv(z.open(f"{t}.txt"), dtype=str, low_memory=False)
    return feed

def active_service_ids(feed, date_str):
    """Service ids running on date_str (YYYY-MM-DD), applying calendar then calendar_dates exceptions."""
    d = pd.Timestamp(date_str); ymd = d.strftime("%Y%m%d"); dow = d.day_name().lower()
    active = set()
    if "calendar" in feed:
        c = feed["calendar"]
        m = (c["start_date"] <= ymd) & (c["end_date"] >= ymd) & (c[dow] == "1")
        active |= set(c.loc[m, "service_id"])
    if "calendar_dates" in feed:
        cd = feed["calendar_dates"]; cd = cd[cd["date"] == ymd]
        active |= set(cd.loc[cd["exception_type"] == "1", "service_id"])
        active -= set(cd.loc[cd["exception_type"] == "2", "service_id"])
    return active

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dl = p2 - p1, np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*r*np.arcsin(np.sqrt(a))

def shape_lengths_km(feed):
    s = feed["shapes"].copy()
    for c in ("shape_pt_lat","shape_pt_lon"): s[c] = s[c].astype(float)
    s["shape_pt_sequence"] = s["shape_pt_sequence"].astype(int)
    s = s.sort_values(["shape_id","shape_pt_sequence"])
    g = s.groupby("shape_id")
    s["seg"] = haversine_km(g["shape_pt_lat"].shift(), g["shape_pt_lon"].shift(), s["shape_pt_lat"], s["shape_pt_lon"])
    return s.groupby("shape_id")["seg"].sum()

def trip_km(feed, trip_ids, dist_unit_m=None):
    """Km per trip. Preferred: shape_dist_traveled (unit must be confirmed: metres or km);
    fallback: geodesic length of the trip's shape. Prints which method each trip used."""
    st = feed["stop_times"][feed["stop_times"]["trip_id"].isin(trip_ids)]
    km = pd.Series(dtype=float)
    if "shape_dist_traveled" in st and st["shape_dist_traveled"].notna().any():
        d = st.assign(x=pd.to_numeric(st["shape_dist_traveled"], errors="coerce")).groupby("trip_id")["x"].agg(lambda v: v.max() - v.min())
        assert dist_unit_m is not None, "Confirm shape_dist_traveled unit (metres or km) and pass dist_unit_m=True/False"
        km = d / 1000 if dist_unit_m else d
    missing = set(trip_ids) - set(km.dropna().index)
    if missing and "shapes" in feed:
        lens = shape_lengths_km(feed)
        t = feed["trips"].set_index("trip_id").loc[list(missing), "shape_id"]
        km = pd.concat([km.dropna(), t.map(lens)])
    print(f"trip_km: {len(km)} trips, {len(missing)} via shape geometry fallback")
    return km

## 1 · Data acquisition
Download every source from its public landing page and log download date, size and checksum. Files that need registration or an API key are obtained manually, placed in `data_raw/` and logged with `manual=True` (brief 5.1). No API keys in this notebook.

Direct file URLs come from the portal's CKAN API (`https://datos.madrid.es/api/3/action/package_show?id=<dataset>`); the files themselves are served by `datos.emtmadrid.es`.

**The GTFS URL is a rolling feed.** EMT regenerates `gtfs_emt.zip` daily and its calendar starts on the generation day, so a later download is a different file. `fetch()` never overwrites a file already in `data_raw/`: keep the snapshot used for the submission and quote its SHA-256 from the register. The feed also has a `frequencies.txt` row for almost every trip, but those rows only describe each hour's headway: the trips in `trips.txt` / `stop_times.txt` are already individual timed runs, chained into vehicle blocks (`block_id`). §4 must **not** expand them; doing so multiplies daily km about 26× (8.8 M km instead of 0.34 M). This is the brief's "avoid double counting repeated trips" rule.

In [ ]:
# Direct file URLs, resolved from each landing page via the portal's CKAN API (brief section 4.1).
EMT = "https://datos.emtmadrid.es/dataset"
SOURCES = {
    # filename: (url, note)
    "emt_gtfs.zip": (f"{EMT}/9b23259a-4491-494b-9695-36a7709b2c12/resource/3cba2058-9833-422c-a704-bf992d31d2ee/download/gtfs_emt.zip",
                     "EMT GTFS, rolling daily feed — https://datos.madrid.es/dataset/900030-0-emt-autobus-gtfs"),
    "emt_supply_2026.csv": (f"{EMT}/03c64a21-3ee8-461a-9f9a-cb4c727a41be/resource/8400b44b-ca21-45b9-b3cf-7a15b754b739/download/2026busoferta.csv",
                     "Buses in service per day, line and hour band, 2026 — https://datos.madrid.es/dataset/300596-0-emt-lineas-autobuses-tramos"),
    "emt_fleet_history.csv": (f"{EMT}/071db69e-5e59-4ceb-a6da-92d895f56b62/resource/48d59bf3-93a8-48b7-96bd-cf78cc7c902b/download/vehicles.csv",
                     "Fleet history (vehicle, dates, fuel, size) — https://datos.madrid.es/dataset/300241-0-emt-flota-autobuses"),
    "emt_calendar.csv": (f"{EMT}/405a6ed0-9464-4d04-9edb-939031717b20/resource/b8cceab8-9173-4eec-8431-21f5e9e25834/download/calendar.csv",
                     "Operating calendar 2024-2027 — https://datos.madrid.es/dataset/900022-0-calendario-emt-autobus"),
    "emt_ridership_2026.csv": (f"{EMT}/29b27217-429f-44a0-a90a-2db7f13529b6/resource/07622d4b-6c64-43ea-a398-69134d6128bc/download/2026demdialineafranjaanual.csv",
                     "Optional: daily riders per line and hour band, 2026 — https://datos.madrid.es/dataset/900028-0-emt-demanda-autobus"),
}
PATHS = {fn: fetch(url, fn, note) for fn, (url, note) in SOURCES.items()}
show_register()

**What each source covers.** Load the small tables and print the first and last date in each file. The checks confirm the columns we rely on and flag the GTFS / supply gap handled in §2.

In [ ]:
# Load the small tables and print what period each source actually covers.
gtfs_cal = read_gtfs(PATHS["emt_gtfs.zip"], tables=("calendar", "calendar_dates", "agency"))
supply = pd.read_csv(PATHS["emt_supply_2026.csv"], sep=";", dtype={"Linea": str})
riders = pd.read_csv(PATHS["emt_ridership_2026.csv"], sep=";", dtype={"Linea": str})
fleet_hist = pd.read_csv(PATHS["emt_fleet_history.csv"], dtype=str).apply(lambda c: c.str.strip())
emt_cal = pd.read_csv(PATHS["emt_calendar.csv"], dtype=str, keep_default_na=False)   # 'NA' = Christmas-period day code, not missing

with zipfile.ZipFile(PATHS["emt_gtfs.zip"]) as z:
    gtfs_files = sorted(n[:-4] for n in z.namelist())
coverage = pd.DataFrame([
    ("emt_gtfs.zip", gtfs_cal["calendar"].start_date.min(), gtfs_cal["calendar"].end_date.max(), ", ".join(gtfs_files)),
    ("emt_supply_2026.csv", supply.FechaSer.min(), supply.FechaSer.max(), f"{supply.Linea.nunique()} lines"),
    ("emt_ridership_2026.csv", riders.Fecha.min(), riders.Fecha.max(), f"{riders.Linea.nunique()} lines"),
    ("emt_fleet_history.csv", fleet_hist.dateIni.min()[:10], fleet_hist.dateIni.max()[:10], f"{len(fleet_hist)} records, {fleet_hist.vehicleId.nunique()} vehicles"),
    ("emt_calendar.csv", emt_cal.Fecha.min(), emt_cal.Fecha.max(), f"{len(emt_cal)} days"),
], columns=["file", "first", "last", "content"])
display(coverage)

check("GTFS has the tables the line model needs", {"routes", "trips", "stop_times", "shapes", "calendar"} <= set(gtfs_files))
check("Supply data has the documented columns", list(supply.columns) == ["Linea", "FechaSer", "TipoDiaMo", "CodFranja", "NumCCReal"])
check("Fleet history has fuel type and size", {"fuelType", "typeCapacity", "dateIni", "dateEnd"} <= set(fleet_hist.columns))
check("Calendar covers 2025-2026", emt_cal.Fecha.min() <= "2025-01-01" and emt_cal.Fecha.max() >= "2026-12-31")
check("Supply data ends before GTFS validity starts (gap handled in section 2)",
      supply.FechaSer.max().replace("-", "") < gtfs_cal["calendar"].start_date.min(),
      f"supply ends {supply.FechaSer.max()}, GTFS starts {gtfs_cal['calendar'].start_date.min()}")

## 2 · Reference day
Rule (brief section 2): a Tuesday or Wednesday, regular working day in EMT's calendar, outside holidays and the summer timetable, no strikes or events, inside GTFS validity.

**How we apply it.** A day is eligible when it is a Tuesday or Wednesday coded `LA` (working day) in the standard season `IV`, **and every weekday of its week is also `LA` and `IV`**, so the week has no holiday, strike (`HU`), event (`EV`) or seasonal timetable around it.
- **Reference day** (`REF_DATE`, used for GTFS km in File 1 and File 2) = the first eligible day inside the GTFS snapshot's validity.
- **Supply day** (`SUPPLY_DATE`, used for `peak_buses`) = the latest eligible day **on the same weekday** in the supply data whose total peak buses (sum of each line's peak) is within a tolerance of the median eligible day. The supply data ends 31 Aug 2026 and the GTFS starts on its download day, so no single day has both; the brief allows the closest equivalent weekday in that case.

**Why the typicality test.** The calendar alone points to Tue 16 Jun 2026, but service and ridership that day are already 4–5 % below a normal Tuesday (the end of the school and university term). A November reference day is a full-service day, so we take the latest Tuesday that still looks like one and report 16 Jun as a sensitivity. Both dates go in the report.

> **Decision record · reference and supply days**
> | Option | Verdict | Why |
> |---|---|---|
> | Calendar rule only (16 Jun 2026) | Rejected, kept as sensitivity | Service already 4.4 % below a normal Tuesday (end of term); −77 peak buses over 89 lines |
> | **Calendar rule + typicality within 3 % (9 Jun 2026)** | **Chosen** | Latest real day that looks like a full-service day, as a November reference day is |
> | Median of all eligible Tuesdays, line by line | Rejected | Smoother, but not a published day; the brief asks for a day's supply data |
> | A Wednesday instead of a Tuesday | Rejected | The supply day must match the reference day's weekday, and 17 Nov is the first eligible day in GTFS validity |

In [ ]:
# Eligible days: Tue/Wed, working day, standard season, in a fully regular Mon-Fri week.
cal = emt_cal.assign(date=pd.to_datetime(emt_cal.Fecha)).sort_values("date")
cal["dow"] = cal.date.dt.dayofweek                               # 0 = Monday
cal["week"] = cal.date.dt.to_period("W-SUN")
cal["regular"] = (cal.TipoDiaEs == "LA") & (cal.TemporadaTG == "IV")
week_ok = cal[cal.dow < 5].groupby("week").regular.all()
cal["regular_week"] = cal.week.map(week_ok).fillna(False).astype(bool)
eligible = cal[cal.dow.isin([1, 2]) & cal.regular & cal.regular_week]

gtfs_start = pd.to_datetime(gtfs_cal["calendar"].start_date.min())
gtfs_end = pd.to_datetime(gtfs_cal["calendar"].end_date.max())
ref = eligible[eligible.date.between(gtfs_start, gtfs_end)].iloc[0]
REF_DATE = assume("Reference day", ref.Fecha, "D",
                  "First Tue/Wed in GTFS validity coded LA + IV with a fully regular Mon-Fri week (EMT calendar)")

# Supply-day candidates: same weekday, same rule, inside the supply data. Typicality = sum of each line's peak buses.
TYPICAL_TOL_PCT = assume("Supply-day typicality tolerance", 3, "A",
                         "Max % gap between a candidate's total peak buses and the median eligible day", "SUPPLY_DATE", "16 Jun shown below")
cand = eligible.loc[(eligible.dow == ref.dow) & (eligible.Fecha <= supply.FechaSer.max()), "Fecha"]
s = supply[supply.FechaSer.isin(set(cand))]
peak_by_day = s.groupby(["FechaSer", "Linea"]).NumCCReal.max().unstack()          # day x line
riders_day = riders[riders.Fecha.isin(set(cand))].groupby("Fecha").TotalViajeros.sum()
typ = pd.DataFrame({"sum_line_peaks": peak_by_day.sum(axis=1), "lines_running": peak_by_day.notna().sum(axis=1),
                    "riders": riders_day})
typ["peaks_vs_median_pct"] = (typ.sum_line_peaks / typ.sum_line_peaks.median() - 1) * 100
typ["riders_vs_median_pct"] = (typ.riders / typ.riders.median() - 1) * 100
typ["typical"] = typ.peaks_vs_median_pct.abs() <= TYPICAL_TOL_PCT
display(typ.round(1).tail(8))

calendar_only = typ.index[-1]                                   # latest candidate by the calendar rule alone
SUPPLY_DATE = assume("Supply-data day for peak_buses", typ.index[typ.typical][-1], "D",
                     "Latest same-weekday eligible day in the supply data within the typicality tolerance (brief: closest equivalent weekday)",
                     "File 2 peak_buses", f"Calendar-only day {calendar_only}")
print(f"GTFS snapshot valid {gtfs_start.date()} to {gtfs_end.date()} | supply data ends {supply.FechaSer.max()}")
print(f"Reference day: {REF_DATE} ({ref.date.day_name()}) | supply day: {SUPPLY_DATE} | calendar-only supply day: {calendar_only}")

# The calendar week around each chosen day.
for d in (REF_DATE, SUPPLY_DATE):
    w = cal[cal.week == cal.loc[cal.Fecha == d, "week"].iloc[0]]
    display(w[["Fecha", "DiaSemana", "TipoDiaEs", "TemporadaTG"]].set_index("Fecha").T)

**Checks on the chosen days.** GTFS must run its working-day service on the reference day, the supply day must be typical line by line, and both days must cover the same lines (lines missing from the supply data get a proxy in §4).

In [ ]:
# Checks: GTFS runs a normal weekday service on REF_DATE; SUPPLY_DATE is typical; the two days cover the same lines.
feed_trips = read_gtfs(PATHS["emt_gtfs.zip"], tables=("calendar", "calendar_dates", "trips"))
services = active_service_ids(feed_trips, REF_DATE)
ref_trips = feed_trips["trips"][feed_trips["trips"].service_id.isin(services)]
ref_lines = set(ref_trips.route_id)
check("GTFS: reference day runs the working-day service", "LA" in services,
      f"services {sorted(services)}, {len(ref_trips)} trip patterns, {len(ref_lines)} lines")

dev = (peak_by_day.loc[SUPPLY_DATE] - peak_by_day.drop(index=SUPPLY_DATE).median()).dropna()
check("Supply day: total peak buses within tolerance of the median eligible day", typ.loc[SUPPLY_DATE, "typical"],
      f"{typ.loc[SUPPLY_DATE, 'peaks_vs_median_pct']:+.1f}% over {len(typ)} candidate days")
check("Supply day: most lines within 1 bus of their usual peak", (dev.abs() <= 1).mean() >= 0.9,
      f"{(dev.abs() <= 1).mean():.0%} of {len(dev)} lines")

sens = (peak_by_day.loc[calendar_only] - peak_by_day.loc[SUPPLY_DATE]).dropna()
print(f"Sensitivity: peak_buses on {calendar_only} instead of {SUPPLY_DATE}: total {sens.sum():+.0f} buses, "
      f"{(sens != 0).sum()} lines change, largest change {sens.abs().max():.0f}")

sup_lines = set(supply.loc[supply.FechaSer == SUPPLY_DATE, "Linea"])
print(f"Lines on the reference day (GTFS): {len(ref_lines)} | on the supply day: {len(sup_lines)} | in both: {len(ref_lines & sup_lines)}")
print("GTFS only (need a peak_buses proxy in section 4):", sorted(ref_lines - sup_lines))
print("Supply only (not in the GTFS snapshot):", sorted(sup_lines - ref_lines))
check("Most reference-day lines have a supply record", len(ref_lines & sup_lines) / len(ref_lines) >= 0.9,
      f"{len(ref_lines & sup_lines)} of {len(ref_lines)}")

## 3 · Fleet baseline at 31 Dec 2025 and 2021–25 pace
Rebuild the fleet in service at 31 Dec 2020 and 31 Dec 2025 from the fleet register (each vehicle's start and end dates), by fuel and size; reconcile the 2025 total against EMT's and the city's published figures; print the CNG age profile, which drives retirements in File 4.

> **Decision record · 2025 baseline** (16–18 Nov 2026)
> | Source | Total | Battery / hydrogen / CNG | Verdict |
> |---|---|---|---|
> | **Fleet register, rebuilt here (vehicle by vehicle)** | **2,210** | **452 / 10 / 1,748** | **Chosen:** reproducible, gives the fuel and size split, and its total matches the city's official figure exactly |
> | City of Madrid, [Carta de Servicios 2026](https://www.madrid.es/UnidadesDescentralizadas/Calidad/CARTAS%20SERVICIOS/SistemaCartasServicio/66%20CS_Transporte%20en%20Autob%C3%BAs%20Urbano/DEFINITIVA/CS_TransporteUrbano_2026.pdf) (data at 31 Dec 2025) | 2,210 | not given | Confirms the total |
> | [Comunidad de Madrid budget release](https://crtm.es/comunicacion/sala-de-prensa/noticias/noticias/03122025-la-comunidad-de-madrid-aprueba-441-5-millones-para-la-emt-en-2026-destinados-a-su-flota-de-2215-autobuses/), 3 Dec 2025 | 2,215 | 79% gas, >20% electric or hydrogen | Earlier date, rounded shares: noted only |
> | EMT, Principales cifras web page (31 Dec 2025) | 2,150 | not given | Lower than both official totals; likely a different count (e.g. buses in service). Noted, not used |
> | EMT 2025 accounts ([press summary](https://vivirediciones.es/madrid/cuentas-2025-cierran-con-beneficio-de-326-millones-y-records-de-viajes/), 25 Mar 2026) | 2,239 implied | 456 / 10 / 1,773 | No published total matches its sum: run as a sensitivity |
>
> **Conflicting records.** Five 2012 Tempus buses (9301–9305) appear as both CNG and electric with identical dates; they count as CNG (not zero-emission), which gives 76 zero-emission buses at 31 Dec 2020. Counting them as electric (81) is shown as a sensitivity: the year of full conversion does not move.
>
> No plug-in CNG hybrids or other fuel codes are in service at 31 Dec 2025, so `n_bev + n_h2 + n_cng = fleet_total` holds without a counting rule. The register's newest entry is 5 Jun 2025; the matching city total suggests no material gap, but the four-bus electric difference with EMT's accounts is reported below.

In [ ]:
# Rebuild the fleet in service on a date from the fleet register: the record(s) valid on that date, one row per vehicle.
reg = fleet_hist.drop_duplicates().copy()
reg["start"] = pd.to_datetime(reg.dateIni.str[:10])
reg["end"] = pd.to_datetime(reg.dateEnd.str[:10].replace("9999-12-31", "2262-04-11"))   # open-ended = still in service
first_in_service = reg.groupby("vehicleId").start.min()

FUEL_CLASS = assume("Fuel classes for the baseline", {"EL": "n_bev", "H": "n_h2", "GN": "n_cng"}, "A",
                    "Fleet register codes: EL battery-electric, H hydrogen fuel cell, GN compressed natural gas", "Files 1, 4")
ZE_CODES = {"EL", "H"}
CONFLICT_RULE = assume("Vehicle with conflicting fuel codes on the same date", "count as not zero-emission", "A",
                       "Conservative: a bus counts as zero-emission only if every record valid on the date says EL or H "
                       "(2012 Tempus buses 9301-9305 carry both GN 'TEMPUS GNC' and EL 'TEMPUS EL' records with identical dates)",
                       "ZE_2020, S0 pace", "Counted as electric: see the pace sensitivity below")

def fleet_at(date):
    """Vehicles in service on `date`, one row per vehicle, with age since first entry and size (code 02 = articulated, section 4)."""
    d = pd.Timestamp(date)
    a = reg[(reg.start <= d) & (reg.end > d)].copy()
    a["conflict"] = a.groupby("vehicleId").fuelType.transform("nunique") > 1
    a["ze_record"] = a.fuelType.isin(ZE_CODES)
    a = a.sort_values(["ze_record", "start"]).drop_duplicates("vehicleId", keep="first")   # non-zero-emission record wins a conflict
    a["age_years"] = (d - a.vehicleId.map(first_in_service)).dt.days / 365.25
    a["size"] = np.where(a.typeCapacity == "02", "Articulated", "Standard")
    return a.drop(columns="ze_record")

f20, f25 = fleet_at("2020-12-31"), fleet_at("2025-12-31")
mix = pd.DataFrame({"31 Dec 2020": f20.fuelType.value_counts(), "31 Dec 2025": f25.fuelType.value_counts()}).fillna(0).astype(int)
mix.loc["total"] = mix.sum()
display(mix)
display(pd.crosstab(f25.fuelType, f25["size"], margins=True, margins_name="total"))
check("2025 fleet: only battery-electric, hydrogen and CNG buses in service", set(f25.fuelType) <= set(FUEL_CLASS),
      f"codes in service: {sorted(set(f25.fuelType))}")
print("Vehicles with conflicting fuel codes on 31 Dec 2020:", sorted(f20.loc[f20.conflict, "vehicleId"]), "| on 31 Dec 2025:", sorted(f25.loc[f25.conflict, "vehicleId"]))
print("Codes in service at 31 Dec 2020 that are neither zero-emission nor CNG:",
      f20.loc[~f20.fuelType.isin(list(FUEL_CLASS)), "fuelType"].value_counts().to_dict())

**Reconciliation with published figures.** The rebuilt total is checked against the city's official figure; EMT's accounts mix is kept as a sensitivity on the pace and the year of full conversion.

In [ ]:
PUBLISHED = pd.DataFrame([
    ("City of Madrid, Carta de Servicios 2026", "data at 31 Dec 2025", 2210, None, None, None, "https://www.madrid.es/UnidadesDescentralizadas/Calidad/CARTAS%20SERVICIOS/SistemaCartasServicio/66%20CS_Transporte%20en%20Autob%C3%BAs%20Urbano/DEFINITIVA/CS_TransporteUrbano_2026.pdf"),
    ("Comunidad de Madrid, EMT 2026 budget", "3 Dec 2025", 2215, None, None, None, "https://crtm.es/comunicacion/sala-de-prensa/noticias/noticias/03122025-la-comunidad-de-madrid-aprueba-441-5-millones-para-la-emt-en-2026-destinados-a-su-flota-de-2215-autobuses/"),
    ("EMT, Principales cifras (web)", "31 Dec 2025", 2150, None, None, None, "https://www.emtmadrid.es/Elementos-Cabecera/Enlaces-Pie-vertical/EMPRESA/Somos/Principales-cifras-(1)"),
    ("EMT 2025 accounts (press summary)", "end of 2025", 456 + 10 + 1773, 456, 10, 1773, "https://vivirediciones.es/madrid/cuentas-2025-cierran-con-beneficio-de-326-millones-y-records-de-viajes/"),
], columns=["source", "date", "fleet_total", "n_bev", "n_h2", "n_cng", "url"])
assume("Published 2025 fleet figures", PUBLISHED[["source", "fleet_total", "n_bev", "n_h2", "n_cng"]].to_dict("records"), "V",
       "City of Madrid Carta de Servicios 2026; Comunidad de Madrid 3 Dec 2025; EMT Principales cifras (web); EMT accounts press summary 25 Mar 2026")
register = dict(source="Fleet register (rebuilt)", date="31 Dec 2025", fleet_total=len(f25),
                n_bev=int((f25.fuelType == "EL").sum()), n_h2=int((f25.fuelType == "H").sum()), n_cng=int((f25.fuelType == "GN").sum()), url="")
recon = pd.concat([pd.DataFrame([register]), PUBLISHED], ignore_index=True)
recon["gap_vs_register"] = recon.fleet_total - register["fleet_total"]
display(recon.drop(columns="url"))
check("Rebuilt 2025 total equals the City of Madrid's official figure", register["fleet_total"] == PUBLISHED.at[0, "fleet_total"],
      f"{register['fleet_total']} vs {PUBLISHED.at[0, 'fleet_total']}")

**CNG age profile.** Older gas buses retire first in File 4; the age bands show how many reach the end of a typical life within the horizon (the life itself is a scenario assumption, set in section 8).

In [ ]:
cng = f25[f25.fuelType == "GN"]
bands = pd.cut(cng.age_years, bins=[0, 4, 8, 12, 16, 20], right=False, labels=["0-4", "4-8", "8-12", "12-16", "16-20"])
age_profile = bands.value_counts().sort_index().rename("CNG buses").to_frame()
age_profile["share"] = (age_profile["CNG buses"] / len(cng)).round(3)
display(age_profile.T)
print(f"CNG buses: {len(cng)} | mean age {cng.age_years.mean():.1f} years | oldest {cng.age_years.max():.1f} years")

**Baseline used by Files 1 and 4.** Taken from the rebuilt register; the compliance cell checks that File 1's Madrid row matches it.

In [ ]:
BASELINE_2025 = dict(fleet_total=register["fleet_total"], n_bev=register["n_bev"], n_h2=register["n_h2"], n_cng=register["n_cng"],
                     std_share=round(float((f25["size"] == "Standard").mean()), 4))
assume("2025 baseline fleet", BASELINE_2025, "D",
       "Fleet register rebuilt at 31 Dec 2025; total matches the City of Madrid's Carta de Servicios 2026", "Files 1, 4",
       "EMT accounts mix (456 / 10 / 1,773)")
ZE_2020 = int(f20.fuelType.isin(ZE_CODES).sum())
S0_PACE_UNADJUSTED = assume("S0 pace, unadjusted (zero-emission buses added per year)",
                            (BASELINE_2025["n_bev"] + BASELINE_2025["n_h2"] - ZE_2020) / 5, "D",
                            "Brief Objective 3: (zero-emission fleet 2025 - zero-emission fleet 2020) / 5", "File 4 S0")
check("Baseline identity: n_bev + n_h2 + n_cng = fleet_total",
      BASELINE_2025["n_bev"] + BASELINE_2025["n_h2"] + BASELINE_2025["n_cng"] == BASELINE_2025["fleet_total"])

def year_at_pace(ze_2025, n_cng, pace):
    return 2025 + int(np.ceil(n_cng / pace))
acc = PUBLISHED.set_index("source").loc["EMT 2025 accounts (press summary)"]
sens = pd.DataFrame([
    dict(case="Register (baseline)", ze_2025=BASELINE_2025["n_bev"] + BASELINE_2025["n_h2"], n_cng=BASELINE_2025["n_cng"]),
    dict(case="EMT accounts mix", ze_2025=int(acc.n_bev + acc.n_h2), n_cng=int(acc.n_cng)),
    dict(case="Conflicting 2020 records counted as electric", ze_2025=BASELINE_2025["n_bev"] + BASELINE_2025["n_h2"], n_cng=BASELINE_2025["n_cng"]),
])
ze_2020_alt = ZE_2020 + int(f20.conflict.sum())
sens["ze_2020"] = [ZE_2020, ZE_2020, ze_2020_alt]
sens["pace_per_year"] = ((sens.ze_2025 - sens.ze_2020) / 5).round(1)
sens["year_100pct_at_pace"] = [year_at_pace(z, c, p) for z, c, p in zip(sens.ze_2025, sens.n_cng, sens.pace_per_year)]
display(sens)
print(f"Baseline 31 Dec 2025: {BASELINE_2025} | zero-emission 2020: {ZE_2020} | unadjusted S0 pace: {S0_PACE_UNADJUSTED:.1f} a year")
# TODO(owner: scenarios) adjusted S0 pace: argue it from tenders and deliveries (e.g. 120 e-buses due 2026-H1 2027) and report both paces in File 4 and the report

## 4 · Line model (File 2 metrics)
Every EMT line with ≥1 trip on the reference day stays in File 2, including night, airport and university lines. Public line codes (`C1`, `N26`) come from GTFS `route_short_name`; GTFS `route_id` (`068`) is the key that joins GTFS, supply and ridership.

**Daily km — method B (primary).** EMT publishes each line's route as a KML file per direction and date. `daily_km` = Σ over directions (KML route length × trips run in that direction on the reference day). Lengths are measured on the route geometry with the haversine formula.
**Method A (validation only).** The same km from GTFS: each trip's `shape_dist_traveled` (metres), summed once per trip. GTFS trips are already individual timed runs chained in vehicle blocks, so `frequencies.txt` is **not** expanded (that would double count about 26×). Lines where A and B disagree are listed; KML is used in File 2 unless a line has no KML for a direction it runs.

> **Decision record · daily km** (team decision, 16 Nov 2026)
> | Option | Verdict | Why |
> |---|---|---|
> | **B · KML route length × trips per direction** | **Chosen for File 2** | EMT's published route geometry, a source the brief lists for route length; one length per line and direction |
> | A · GTFS `shape_dist_traveled` per trip | Kept as validation | Independent check on the same day: agrees within 0.7 % city-wide |
> | Expand `frequencies.txt` headway rows | Rejected | Trips are already individual timed runs; expanding gives 8.8 M km/day instead of 0.34 M (double counting, brief §4.1) |
> | KML `distancia` field instead of geometry | Rejected | Field is a cumulative position along the route, not a segment length; geometry matches GTFS (line 1: 8.92 vs 8.82 km) |

In [ ]:
# Method B: KML route length per line and direction on REF_DATE (EMT open API, no key needed).
import time, xml.etree.ElementTree as ET
KML_NS = "{http://www.opengis.net/kml/2.2}"
KML_DIR = os.path.join(RAW, "emt_kml", REF_DATE)
os.makedirs(KML_DIR, exist_ok=True)
KML_URL = "https://openapi.emtmadrid.es/v1/bus/kml/{date}/{line}/{direction}/3"      # type 3 = route geometry

def kml_route_km(route_id, direction):
    """Length in km of EMT's published route for a line (GTFS route_id) and direction (1 = ida, 2 = vuelta); NaN if none."""
    path = os.path.join(KML_DIR, f"L{route_id}_D{direction}.kml")
    if not os.path.exists(path):
        url = KML_URL.format(date=pd.Timestamp(REF_DATE).strftime("%d-%m-%Y"), line=int(route_id), direction=direction)
        r = requests.get(url, timeout=60)
        time.sleep(0.1)
        if r.status_code != 200 or b"<coordinates>" not in r.content:
            return np.nan
        with open(path, "wb") as f:
            f.write(r.content)
    km = 0.0
    for ls in ET.parse(path).getroot().iter(f"{KML_NS}LineString"):
        xy = np.array([[float(v) for v in pt.split(",")[:2]] for pt in ls.find(f"{KML_NS}coordinates").text.split()])
        km += haversine_km(xy[:-1, 1], xy[:-1, 0], xy[1:, 1], xy[1:, 0]).sum()
    return km

routes = read_gtfs(PATHS["emt_gtfs.zip"], tables=("routes",))["routes"].set_index("route_id")
trips_dir = ref_trips.groupby(["route_id", "direction_id"]).size().unstack(fill_value=0)   # trips per line and direction
kml = pd.DataFrame({d: [kml_route_km(r, d) for r in trips_dir.index] for d in (1, 2)}, index=trips_dir.index)

files = sorted(os.listdir(KML_DIR))
blob = b"".join(open(os.path.join(KML_DIR, f), "rb").read() for f in files)
DATA_REGISTER.append(dict(file=f"emt_kml/{REF_DATE}/ ({len(files)} files)", url=KML_URL, downloaded=RUN_DATE, bytes=len(blob),
                          sha256_12=hashlib.sha256(blob).hexdigest()[:12], obtained="script",
                          note="EMT route KML per line and direction — https://datos.madrid.es/dataset/900025-0-lineas-emt-kml"))
print(f"KML routes found: {kml.notna().sum().sum()} of {kml.size} line-directions ({len(files)} files cached)")
display(kml.describe().round(2))

**Method A and the direction match.** GTFS gives each trip's length; the median per line and direction is the GTFS route length. KML direction 1/2 is matched to GTFS `direction_id` 0/1 line by line, choosing the pairing whose lengths agree best (both directions are often similar, so the choice rarely changes km). Lines missing a KML direction they run fall back to method A.

In [ ]:
# Method A: GTFS km per trip from shape_dist_traveled (metres; line 1 = 8.8 km per trip), once per trip.
feed_km = read_gtfs(PATHS["emt_gtfs.zip"], tables=("trips", "stop_times", "shapes"))
DIST_UNIT_M = assume("GTFS shape_dist_traveled unit", "metres", "D", "Line 1 trips = 8.8 km, matching its 8.9 km KML route")
trip_len = trip_km(feed_km, set(ref_trips.trip_id), dist_unit_m=True).rename("km")
t = ref_trips.join(trip_len, on="trip_id")
gtfs_len = t.groupby(["route_id", "direction_id"]).km.median().unstack()            # GTFS route length per direction
daily_km_A = t.groupby("route_id").km.sum()

# Match KML direction 1/2 to GTFS 0/1 per line.
same = (kml[1] - gtfs_len["0"]).abs().fillna(0) + (kml[2] - gtfs_len["1"]).abs().fillna(0)
swap = (kml[1] - gtfs_len["1"]).abs().fillna(0) + (kml[2] - gtfs_len["0"]).abs().fillna(0)
len0 = np.where(swap < same, kml[2], kml[1])
len1 = np.where(swap < same, kml[1], kml[2])
km_B = pd.Series(trips_dir.get("0", 0) * len0 + trips_dir.get("1", 0) * len1, index=trips_dir.index)
missing_B = (trips_dir.get("0", 0).gt(0) & pd.isna(len0)) | (trips_dir.get("1", 0).gt(0) & pd.isna(len1))

daily_km = km_B.where(~missing_B, daily_km_A)
km_source = pd.Series(np.where(missing_B, "GTFS (method A)", "KML (method B)"), index=daily_km.index)
cmp = pd.DataFrame({"line": routes.route_short_name.reindex(daily_km.index), "km_B_kml": km_B, "km_A_gtfs": daily_km_A,
                    "diff_pct": (km_B / daily_km_A - 1) * 100, "source": km_source})
print(f"Daily km on {REF_DATE}: method B {km_B.sum():,.0f} | method A {daily_km_A.sum():,.0f} | File 2 uses {daily_km.sum():,.0f}")
print("Lines falling back to method A:", list(cmp.loc[missing_B, "line"]) or "none")
display(cmp.dropna().reindex(cmp.diff_pct.abs().sort_values(ascending=False).index).head(10).round(1))
check("Daily km: methods A and B agree within 3% city-wide", abs(km_B[~missing_B].sum() / daily_km_A[~missing_B].sum() - 1) <= 0.03,
      f"{(km_B[~missing_B].sum() / daily_km_A[~missing_B].sum() - 1) * 100:+.1f}%")
check("Daily km: most lines agree within 5%", (cmp.diff_pct.abs() <= 5).mean() >= 0.9,
      f"{(cmp.diff_pct.abs() <= 5).mean():.0%} of lines")
check("Daily km: every reference-day line has a value", daily_km.notna().all() and len(daily_km) == len(ref_lines))

**Peak buses.** `peak_buses` = maximum buses in service across the hour bands on the supply day, from the official supply data only (brief File 2). The two lines with no supply record take a documented proxy: their number of GTFS vehicle blocks on the reference day, which tracks `peak_buses` closely on every other line.

In [ ]:
peak = supply[supply.FechaSer == SUPPLY_DATE].groupby("Linea").NumCCReal.max()
blocks = ref_trips.groupby("route_id").block_id.nunique()
lines = pd.DataFrame({"line_id": routes.route_short_name.reindex(daily_km.index), "daily_km": daily_km,
                      "peak_buses": peak.reindex(daily_km.index), "blocks": blocks})
r = lines.dropna()[["peak_buses", "blocks"]].corr().iloc[0, 1]
no_supply = lines.peak_buses.isna()
lines.loc[no_supply, "peak_buses"] = lines.loc[no_supply, "blocks"]
lines["peak_buses"] = lines.peak_buses.astype(int)
assume("peak_buses proxy for lines without supply records", dict(zip(lines.loc[no_supply, "line_id"], lines.loc[no_supply, "peak_buses"])),
       "P", f"GTFS vehicle blocks on {REF_DATE}; blocks vs peak buses r = {r:.2f} on the other lines", "File 2, 3")
print(f"Peak buses on {SUPPLY_DATE}: {int(lines.peak_buses.sum())} across {len(lines)} lines | blocks vs peak buses r = {r:.2f}")
display(lines.loc[no_supply])
check("peak_buses: every line has a positive integer", (lines.peak_buses > 0).all())

**Bus type per line — evidence first (alternative A).** EMT does not publish bus type by line, so we start from what is evidenced and fill the rest with a proxy:
1. **How many articulated buses exist.** In the fleet history, size code `02` is the only one whose models are articulated (`…-ART`, 118–122 places); codes `03`, `04`, `06` are midi and mini buses, counted as Standard (the brief has only two types). This fixes the articulated share of the 31 Dec 2025 fleet.
2. **Which lines are evidenced.** Line 34 runs articulated buses: the brief cites EMT's tender for 120 electric buses as linking articulated vehicles, line 34 and Carabanchel, and the 30 articulated Solaris Urbino 18 from that tender go to Carabanchel ([electrive, 31 Mar 2026](https://www.electrive.com/es/2026/03/31/emt-madrid-incorpora-120-autobuses-electricos-de-bateria/)).
3. **Proxy for the rest.** The remaining articulated buses go to the lines with the most riders per peak bus on the supply day (articulated buses are deployed where demand per vehicle is highest) until the articulated share of peak buses matches the fleet's. Labelled **P**.

> **Decision record · bus type** (team decision, 16 Nov 2026)
> | Option | Verdict | Why |
> |---|---|---|
> | **A · Evidence first (fleet size codes + line 34), proxy for the rest** | **Chosen** | Uses every published fact before any assumption; articulated share fixed by the fleet history |
> | B · Bracket low / base / high | Kept as variance check | Shows the proxy barely matters: energy −1.1 % / +1.5 %, no charging strategy changes |
> | All Standard | Rejected | Ignores 85 articulated buses in the 2025 fleet |
> | Assign by depot (e.g. all Carabanchel lines articulated) | Rejected for now | Depot allocation is not done yet (§5); revisit if B finds a depot-level source |

In [ ]:
fl = fleet_hist.assign(cap=pd.to_numeric(fleet_hist.capacity, errors="coerce")).drop_duplicates()
fl = fl.sort_values("cap", ascending=False).drop_duplicates("vehicleId")             # one record per vehicle (notebook 03 rule)
in_service = fl[(fl.dateIni <= "2025-12-31 23:59:59") & (fl.dateEnd > "2025-12-31 23:59:59")]
size_codes = in_service.groupby("typeCapacity").agg(buses=("vehicleId", "count"), min_places=("cap", "min"), max_places=("cap", "max"),
                                                    models=("model", lambda m: ", ".join(sorted(set(m)))))
display(size_codes)
check("Size code 02 = articulated models only", in_service.loc[in_service.typeCapacity == "02", "model"].str.contains("ART").all())

ART_SHARE = assume("Articulated share of the 31 Dec 2025 fleet", round((in_service.typeCapacity == "02").mean(), 4), "D",
                   "Fleet history: size code 02 (models ...-ART) / buses in service")
EVIDENCED_ART = assume("Lines with evidenced articulated buses", ["034"], "V",
                       "Brief 4.2: EMT tender for 120 e-buses links articulated vehicles, line 34 and Carabanchel; electrive 31 Mar 2026")
art_target = round(ART_SHARE * lines.peak_buses.sum())

riders_sup = riders[riders.Fecha == SUPPLY_DATE].groupby("Linea").TotalViajeros.sum()
lines["riders_per_peak_bus"] = (riders_sup.reindex(lines.index) / lines.peak_buses).round(0)

def articulated_lines(target):
    """Evidenced lines first, then lines by riders per peak bus until articulated peak buses reach `target`."""
    chosen, n = list(EVIDENCED_ART), lines.loc[EVIDENCED_ART, "peak_buses"].sum()
    for rid in lines.drop(index=EVIDENCED_ART).riders_per_peak_bus.dropna().sort_values(ascending=False).index:
        if n + lines.at[rid, "peak_buses"] > target:
            continue
        chosen.append(rid); n += lines.at[rid, "peak_buses"]
    return chosen

ART_BASE = articulated_lines(art_target)
lines["bus_type"] = np.where(lines.index.isin(ART_BASE), "Articulated", "Standard")
assume("Articulated lines (proxy)", list(lines.loc[ART_BASE, "line_id"]), "P",
       f"Line 34 evidenced; others = highest riders per peak bus on {SUPPLY_DATE} until {ART_SHARE:.1%} of peak buses", "File 2, 3", "Bracket below")
print(f"Articulated: {len(ART_BASE)} lines, {lines.loc[ART_BASE, 'peak_buses'].sum()} peak buses (target {art_target})")
display(lines.loc[ART_BASE, ["line_id", "peak_buses", "riders_per_peak_bus", "daily_km"]].round(0))

**File 2 formulas (brief Rule 2 and File 2 definitions).** km per bus-day, energy per bus-day, the charging strategy and annual CO₂ avoided, all from `P`.

**Charging strategy above range.** Rule 2 fixes Depot (km per bus-day within 300 km Standard / 240 km Articulated) and leaves Opportunity vs Not_feasible to the team. Our rule is about energy: a bus above range must get the difference, `top-up = (km per bus-day − range) × kWh/km`, from daytime charging. While that top-up is at most one more battery's worth of driving (km per bus-day ≤ 2 × range), terminus or mid-day depot charging can plausibly supply it: **Opportunity**. Beyond that, the bus would need more energy during the day than it gets overnight, which is a fleet-size problem rather than a charging one: **Not_feasible** (more buses on the line, not more chargers).

> **Decision record · Opportunity vs Not_feasible above range** (18 Nov 2026)
> | Option | Verdict | Why |
> |---|---|---|
> | **Energy rule: Opportunity if km per bus-day ≤ 2 × range, else Not_feasible** | **Chosen** | Uses only fixed brief parameters plus one labelled multiplier; reproducible; the top-up energy it reports sizes terminus chargers in §6 / §8 |
> | Same rule with 1.5 × range | Kept as sensitivity | Stricter; shows which lines flip (next cells) |
> | Dwell-time rule (top-up must fit in scheduled terminus layovers) | Rejected for now | EMT's GTFS leaves almost no time between a bus's trips (median 0.2 min, below), so layovers are not visible in the timetable; recovery time is likely inside trip times. Revisit if EMT publishes layovers or a charger location list |
> | Every above-range line Not_feasible | Rejected | Hides the daytime top-up need from the capex, though a charger can close a small gap |
> | Every above-range line Opportunity | Rejected | No limit: a line needing several batteries a day is not solved by a charger |
>
> Opportunity is screening-level: whether each terminus has space, grid and timetable slack for a charger remains to be studied before investment (brief, power-assessment protocol item 4).

In [ ]:
OPP_MAX_RANGE_MULT = assume("Opportunity limit: max km per bus-day as a multiple of range", 2.0, "A",
                            "Top-up at most one extra battery of driving a day; above that the line needs more buses, not chargers",
                            "File 2 charging_strategy", "1.5 x range (next cells)")

def line_metrics(lines, opp_mult=OPP_MAX_RANGE_MULT):
    """lines: DataFrame with line_id, daily_km, peak_buses, bus_type. Applies the brief's fixed formulas and our above-range rule."""
    df = lines.copy()
    rng, kwh = df.bus_type.map(P["range_km"]), df.bus_type.map(P["kwh_km"])
    df["km_per_bus_day"] = df.daily_km / df.peak_buses
    df["energy_kwh_bus_day"] = df.km_per_bus_day * kwh
    df["charging_strategy"] = np.select([df.km_per_bus_day <= rng, df.km_per_bus_day <= opp_mult * rng],
                                        ["Depot", "Opportunity"], "Not_feasible")
    df["topup_kwh_bus_day"] = (df.km_per_bus_day - rng).clip(lower=0) * kwh      # battery energy needed during the day
    cng_kg_co2_km = df.bus_type.map(P["cng_kg_km"]) * P["cng_kgco2_per_kg"]
    ele_kg_co2_km = kwh / P["charge_eff"] * P["grid_gco2_kwh"] / 1000
    df["co2_avoided_t_year"] = df.daily_km * P["service_days"] * (cng_kg_co2_km - ele_kg_co2_km) / 1000
    return df

**Bracket (alternative B): how much does the bus-type proxy matter?** The same line model run three ways: **low** = only the evidenced line 34 articulated; **base** = the evidence-first assignment above; **high** = articulated buses on twice as many peak buses, following the same ranking. If charging strategies and energy barely move, the proxy does not drive the conclusions.

In [ ]:
def bracket_case(name, art):
    m = line_metrics(lines.assign(bus_type=np.where(lines.index.isin(art), "Articulated", "Standard")))
    return m, dict(case=name, articulated_lines=len(art), articulated_peak_buses=int(m.loc[art, "peak_buses"].sum()),
                   night_energy_mwh=(m.peak_buses * m.energy_kwh_bus_day).sum() / 1000,
                   lines_above_range=int((m.charging_strategy != "Depot").sum()), co2_avoided_kt=m.co2_avoided_t_year.sum() / 1000)

cases = {"low": EVIDENCED_ART, "base": ART_BASE, "high": articulated_lines(2 * art_target)}
results = {k: bracket_case(k, v) for k, v in cases.items()}
bracket = pd.DataFrame([r[1] for r in results.values()]).set_index("case")
bracket["energy_vs_base_pct"] = (bracket.night_energy_mwh / bracket.at["base", "night_energy_mwh"] - 1) * 100
display(bracket.round(1))
base_cs = results["base"][0].charging_strategy
for k in ("low", "high"):
    flip = results[k][0].charging_strategy != base_cs
    print(f"{k}: {flip.sum()} lines change charging strategy vs base: {list(lines.loc[flip[flip].index, 'line_id'])}")

**Line table so far.** The base case through the File 2 formulas. Depot, priority rank and electrification year are added in §5 and §7, then File 2 is written.

In [ ]:
line_table = line_metrics(lines)
display(line_table[["line_id", "daily_km", "peak_buses", "km_per_bus_day", "bus_type", "energy_kwh_bus_day",
                    "charging_strategy", "co2_avoided_t_year"]].round(1).sort_values("km_per_bus_day", ascending=False).head(15))
print(line_table.charging_strategy.value_counts().to_string())
print(f"Total daily km {line_table.daily_km.sum():,.0f} | km per bus-day: median {line_table.km_per_bus_day.median():.0f}, max {line_table.km_per_bus_day.max():.0f}")

**Above-range lines: top-up need, layover evidence and the 1.5 × sensitivity.** For each line above range: the daytime top-up per bus, and how long that takes at an illustrative terminus charger. The timetable check shows why a dwell-time rule is not used: the gap between one trip's last arrival and the same bus's next departure is close to zero in EMT's GTFS.

In [ ]:
OPP_KW = assume("Illustrative terminus charger power (kW)", 300, "A",
                "Order of magnitude of pantograph terminus chargers; to be sourced before submission", "Minutes shown only, not File 2", "150 kW")
with zipfile.ZipFile(PATHS["emt_gtfs.zip"]) as z:
    st = pd.read_csv(z.open("stop_times.txt"), dtype=str, usecols=["trip_id", "arrival_time", "departure_time", "stop_sequence"])
st = st[st.trip_id.isin(t.trip_id)].assign(seq=lambda d: d.stop_sequence.astype(int)).sort_values("seq")
to_min = lambda s: s.str.split(":", expand=True).astype(int).pipe(lambda d: d[0] * 60 + d[1] + d[2] / 60)
g = st.groupby("trip_id")
trip_times = pd.DataFrame({"start": to_min(g.departure_time.first()), "end": to_min(g.arrival_time.last())})
tb = t.join(trip_times, on="trip_id").sort_values(["block_id", "start"])
tb["gap_min"] = tb.groupby("block_id").start.shift(-1) - tb.end
print(f"Gap between a bus's consecutive trips on {REF_DATE}: median {tb.gap_min.median():.1f} min, "
      f"share of gaps of 5 min or more {(tb.gap_min >= 5).mean():.1%}")

above = line_table[line_table.charging_strategy != "Depot"].copy()
above["range_km"] = above.bus_type.map(P["range_km"])
above["km_vs_range"] = above.km_per_bus_day / above.range_km
for kw in (OPP_KW, 150):
    above[f"topup_min_{kw}kW"] = above.topup_kwh_bus_day / (kw * P["charge_eff"]) * 60
above["strategy_1.5x"] = line_metrics(lines, opp_mult=1.5).loc[above.index, "charging_strategy"]
display(above[["line_id", "bus_type", "peak_buses", "km_per_bus_day", "km_vs_range", "topup_kwh_bus_day",
               f"topup_min_{OPP_KW}kW", "topup_min_150kW", "charging_strategy", "strategy_1.5x"]].round(1)
        .sort_values("km_vs_range", ascending=False))
flip = above[above.charging_strategy != above["strategy_1.5x"]]
print(f"At 1.5 x range, {len(flip)} line(s) change to Not_feasible: {list(flip.line_id)}")
print(f"Daytime top-up across above-range lines: {(above.topup_kwh_bus_day * above.peak_buses).sum() / 1000:.1f} MWh a day on {int(above.peak_buses.sum())} buses")
check("Rule 2: Depot if and only if km per bus-day is within range",
      ((line_table.charging_strategy == "Depot") == (line_table.km_per_bus_day <= line_table.bus_type.map(P["range_km"]))).all())
check("Charging strategy uses only the brief's three labels", set(line_table.charging_strategy) <= {"Depot", "Opportunity", "Not_feasible"})
# Line A (university) is furthest above range because its June peak buses are about half of term time; see the university sensitivity below.

**Data-quality flags on the File 2 inputs (ML I study guide: IQR rule, isolation forest).** Two detectors, one per variable and one across variables:
- **IQR rule** on each input: below Q1 − 1.5·IQR or above Q3 + 1.5·IQR.
- **Isolation forest** on all inputs together, after a log transform (the inputs are right-skewed) and standardisation (so km and buses weigh the same).

Flags are for **review only: no line is deleted or changed**, because every line active on the reference day must stay in File 2 and large lines are real. Missing riders (lines without ridership records) are filled with the median for the detector only.

In [ ]:
from sklearn.ensemble import IsolationForest
FLAG_COLS = ["daily_km", "peak_buses", "km_per_bus_day", "riders_per_peak_bus"]
IQR_K = assume("IQR fence multiplier", 1.5, "A", "Tukey fence (ML I study guide)", "Review flags only")
ISO_SHARE = assume("Isolation forest expected outlier share", 0.05, "A", "About 12 of 234 lines sent to review", "Review flags only")

X = line_table[FLAG_COLS]
def iqr_flag(s):
    q1, q3 = s.quantile([0.25, 0.75])
    return (s < q1 - IQR_K * (q3 - q1)) | (s > q3 + IQR_K * (q3 - q1))
iqr = pd.DataFrame({c: iqr_flag(X[c]) for c in FLAG_COLS})
Z = np.log1p(X.fillna(X.median()))                       # log: right-skewed inputs
Z = (Z - Z.mean()) / Z.std()                             # standardise: mean 0, SD 1
line_table["iqr_flags"] = iqr.apply(lambda r: ", ".join(c for c in FLAG_COLS if r[c]), axis=1)
line_table["iso_outlier"] = IsolationForest(n_estimators=300, contamination=ISO_SHARE, random_state=0).fit(Z).predict(Z) == -1

review = line_table[(line_table.iqr_flags != "") | line_table.iso_outlier]
print(f"Lines flagged for review: {len(review)} of {len(line_table)} (IQR: {(line_table.iqr_flags != '').sum()}, isolation forest: {line_table.iso_outlier.sum()})")
display(review[["line_id"] + FLAG_COLS + ["iqr_flags", "iso_outlier"]].round(1).sort_values("km_per_bus_day", ascending=False))

# Known cause to check: university lines have June peak buses (exam period) but November km (full term).
UNIVERSITY = ["A", "E", "F", "G", "H", "U"]
uni = line_table.index[line_table.line_id.isin(UNIVERSITY)]
print("University lines, peak buses on each eligible supply-data Tuesday (term vs June):")
display(peak_by_day[[r for r in uni if r in peak_by_day.columns]].rename(columns=line_table.line_id).tail(10))
check("Outlier review: no line removed", len(line_table) == len(lines))
# Decision (team, 16 Nov 2026): option (c) — File 2 keeps 9 Jun for every line; the term-time day for university lines is a sensitivity (next cell).

**Sensitivity · university lines' supply day.** On 9 Jun the six university lines (A, E, F, G, H, U) run about half their term-time buses, while their km comes from a November term day.

> **Decision record · university lines** (team decision, 16 Nov 2026)
> | Option | Verdict | Why |
> |---|---|---|
> | (a) 9 Jun for every line | Rejected as the only view | Simplest, but leaves the June/November mismatch unexamined |
> | (b) Term-time Tuesday for the six lines in File 2 | Rejected for File 2 | Mixes two supply days in one file; harder to defend as "the closest equivalent weekday" |
> | **(c) 9 Jun in File 2, term-time day as a sensitivity** | **Chosen** | File 2 follows one rule for every line; the report shows how much the mismatch matters |

The term-time day is picked by rule: the latest eligible Tuesday on which the university lines' total peak buses is within the typicality tolerance of their median across eligible days.

In [ ]:
uni_ids = [r for r in uni if r in peak_by_day.columns]
uni_total = peak_by_day[uni_ids].sum(axis=1)
uni_typical = (uni_total / uni_total.median() - 1).abs() * 100 <= TYPICAL_TOL_PCT
UNI_TERM_DAY = assume("Term-time supply day for university lines (sensitivity)", uni_total.index[uni_typical][-1], "D",
                      "Latest eligible Tuesday with university-line peak buses within tolerance of their median", "Sensitivity only")
alt = lines.copy()
alt.loc[uni_ids, "peak_buses"] = peak_by_day.loc[UNI_TERM_DAY, uni_ids].astype(int)
base_u, alt_u = line_metrics(lines).loc[uni_ids], line_metrics(alt).loc[uni_ids]
uni_sens = pd.DataFrame({"line_id": base_u.line_id, "daily_km": base_u.daily_km,
                         f"peak_{SUPPLY_DATE}": base_u.peak_buses, f"peak_{UNI_TERM_DAY}": alt_u.peak_buses,
                         "km_per_bus_base": base_u.km_per_bus_day, "km_per_bus_term": alt_u.km_per_bus_day,
                         "strategy_base": base_u.charging_strategy, "strategy_term": alt_u.charging_strategy})
display(uni_sens.round(1))
e = lambda m: (m.peak_buses * m.energy_kwh_bus_day).sum() / 1000
print(f"Term-time day {UNI_TERM_DAY}: university peak buses {int(base_u.peak_buses.sum())} -> {int(alt_u.peak_buses.sum())}; "
      f"lines changing charging strategy: {list(uni_sens.loc[uni_sens.strategy_base != uni_sens.strategy_term, 'line_id'])}; "
      f"city night energy {e(line_metrics(lines)):.1f} -> {e(line_metrics(alt)):.1f} MWh (km and CO2 avoided unchanged)")
check("University sensitivity: File 2 inputs unchanged (option c)", lines.loc[uni_ids, "peak_buses"].equals(line_table.loc[uni_ids, "peak_buses"]))

**Per-bus sensitivity (Rule 2).** `km_per_bus_day` is an average; some buses run much longer days. Each GTFS vehicle block (`block_id`) is one bus's planned day on the reference day, so summing its trips' km gives km per bus. A line can be **Depot on average** yet have buses that exceed range: those buses need a daytime top-up or a mid-day swap. This is a sensitivity; File 2 keeps the brief's line-average screening.

Block km uses GTFS trip lengths (method A), the only per-trip source; at city level it is within 1 % of the KML km used in File 2. Deadheading to and from the depot is not in the timetable, so these are in-service km: the true bus-day is somewhat longer.

> **Decision record · day-to-day range** (team decision, 16 Nov 2026)
> | Option | Verdict | Pros | Cons |
> |---|---|---|---|
> | **Vehicle blocks (GTFS `block_id`)** | **Chosen** | EMT's own planned bus days; already in the data; matches peak buses (2,106 blocks vs 2,032, r = 0.98); finds 74 buses (3.5 %) over their range on 26 lines, 19 of which pass on the line average (44 buses); counts buses needing a midday charge or swap | Planned, not operated (mid-day swaps make real days shorter); excludes depot deadheading; no interlining visible |
> | First-to-last-trip span × average speed (brief's example) | Rejected | Simple, brief-suggested | Assumes one bus works the whole span, so it overstates |
> | Daily km ÷ off-peak buses | Rejected | Uses the official supply data | Ignores how buses are actually scheduled |

In [ ]:
# Per-bus range from vehicle blocks: km per block = sum of its trips' GTFS km on REF_DATE.
blk = t.groupby(["route_id", "block_id"]).km.sum().rename("block_km").reset_index()
blk["bus_type"] = blk.route_id.map(lines.bus_type)
blk["range_km"] = blk.bus_type.map(P["range_km"])
blk["over_range"] = blk.block_km > blk.range_km

per_line = blk.groupby("route_id").agg(blocks=("block_id", "count"), blocks_over_range=("over_range", "sum"),
                                       longest_block_km=("block_km", "max"), median_block_km=("block_km", "median"))
per_line["share_over_range"] = per_line.blocks_over_range / per_line.blocks
per_line = line_table[["line_id", "bus_type", "km_per_bus_day", "charging_strategy"]].join(per_line)

hidden = per_line[(per_line.charging_strategy == "Depot") & (per_line.blocks_over_range > 0)]
print(f"Blocks on {REF_DATE}: {len(blk)} | over range: {int(blk.over_range.sum())} ({blk.over_range.mean():.1%}) on "
      f"{(per_line.blocks_over_range > 0).sum()} lines | line average flags {(line_table.charging_strategy != 'Depot').sum()} lines")
print(f"Depot on average but with buses over range: {len(hidden)} lines, {int(hidden.blocks_over_range.sum())} buses")
display(per_line[per_line.blocks_over_range > 0].sort_values("blocks_over_range", ascending=False)
        [["line_id", "bus_type", "charging_strategy", "km_per_bus_day", "blocks", "blocks_over_range",
          "share_over_range", "median_block_km", "longest_block_km"]].round(2).head(20))
print("Block km distribution:", blk.block_km.describe().round(0).to_dict())
check("Per-bus: every reference-day line has blocks", per_line.blocks.notna().all())
check("Per-bus: block km within 1% of method A daily km", abs(blk.block_km.sum() / daily_km_A.sum() - 1) <= 0.01)
# TODO(owner: line model) add deadheading as a sensitivity (depot-to-terminal distance from §5) once depots are allocated
# TODO(owner: scenarios) buses over range need opportunity charging or a mid-day swap: carry their count into §8 capex / fleet

## 5 · Depot allocation and fleet reconciliation
Each line gets one **primary depot** by a transparent rule, with a capacity check, compared with three alternatives and the 2021 per-depot counts. Active buses at a depot are the sum of its lines' peak buses (this is File 3's `buses_assigned`, never inflated by reserves). The physical fleet per depot is reconciled to the 2,210-bus baseline with a reserve ratio r: Fleet_d = Active_d / (1 − r) (brief 4.2, steps 1–5).

> **Decision record · depot allocation** (18–23 Nov 2026)
> | Option | Verdict | Why |
> |---|---|---|
> | **Nearest terminal + capacity check** | **Chosen** | Buses start and end their day at a terminal, so the closest terminal is the natural pull-out point. Pure nearest piles 637 active buses on central La Elipa (2021: 314) and leaves Sanchinarro 124, so a capacity check moves the lines with the cheapest alternative until no depot is more than 10 % above its 2021 size scaled to today's fleet |
> | A · Nearest terminal, no capacity check | Rejected, shown | Ignores depot size; La Elipa would need twice its 2021 fleet |
> | B · Nearest line centre + capacity check | Kept as sensitivity | Uses where the line runs, not where buses start; moves 67 lines |
> | C · La Elipa closed | Kept as sensitivity | La Elipa was closed for rebuilding from March 2022; EMT lists it as an operating centre at 31 Dec 2025, so it is in the base case |
> | Fractional (multi-depot) allocation | Not used | The brief allows one dominant depot; lines whose two nearest depots are within 1 km are listed as "either is plausible" |
> | Las Tablas | Excluded | Announced site (≈620 buses), not an operating 2025 depot (brief 4.2); for File 4 scenarios only |
>
> Distances are straight-line km from GTFS terminal stops to the depot point; road distance for deadheading is a later sensitivity.

In [ ]:
EMT_SYLLABUS = fetch("https://www.emtmadrid.es/getattachment/176260da-71d6-42d2-ac3b-8fdccbf33556/3-Temario-examen-teorico-31-03-2025.aspx",
                     "emt_temario_2025.pdf", note="EMT 2025 recruitment syllabus: addresses of the five operations centres (brief 4.2)")
DEPOTS = pd.DataFrame([
    # depot, address (EMT syllabus 2025), lat, lon (OpenStreetMap feature), coordinate source, EMT web map link (lat, lon)
    ("Carabanchel", "C/ Ventura Díaz Bernardo s/n (esq. Avda. de los Poblados), 28054", 40.371370, -3.724536, "OSM feature 'EMT - Depósito de Carabanchel'", 40.3714373, -3.7284428),
    ("Fuencarral",  "C/ Mauricio Legendre 42, 28046",                                  40.480223, -3.681902, "OSM feature 'EMT - Depósito de Fuencarral'",  40.4778503, -3.6844606),
    ("La Elipa",    "Avda. de las Trece Rosas 3, 28017",                               40.420145, -3.649736, "OSM feature 'EMT - Depósito de La Elipa'",    40.4206425, -3.6508343),
    ("Entrevías",   "Avda. de Santa Catalina 11, 28053",                               40.375129, -3.677643, "OSM street geocode of the address (no named depot feature)", 40.3737054, -3.6793263),
    ("Sanchinarro", "Avda. de Niceto Alcalá Zamora 4, 28050",                          40.483769, -3.646270, "OSM feature 'EMT - Cocheras de Sanchinarro'", 40.4834317, -3.6505758),
], columns=["depot_name", "address", "latitude", "longitude", "coord_source", "emt_map_lat", "emt_map_lon"])
DEPOTS["check_m"] = (haversine_km(DEPOTS.latitude, DEPOTS.longitude, DEPOTS.emt_map_lat, DEPOTS.emt_map_lon) * 1000).round(0)
DEPOTS["label"] = np.where(DEPOTS.coord_source.str.startswith("OSM feature"), "D", "P")
for _, d in DEPOTS.iterrows():
    assume(f"Depot location: {d.depot_name}", (d.latitude, d.longitude), d.label,
           f"Address: EMT 2025 syllabus; point: {d.coord_source}; {d.check_m:.0f} m from EMT's own map link (Principales cifras page)", "File 3, §5")
display(DEPOTS.drop(columns=["emt_map_lat", "emt_map_lon"]))
check("Depot points within 500 m of EMT's own map links", (DEPOTS.check_m <= 500).all(), f"max {DEPOTS.check_m.max():.0f} m")
check("Five 2025 operations centres, Las Tablas excluded (future site, brief 4.2)",
      len(DEPOTS) == 5 and "Las Tablas" not in set(DEPOTS.depot_name))

**Allocation rule and reconciliation.** Primary rule, the depot summary against the 2021 counts, and the reserve-ratio reconciliation of active versus physical fleet. The capacity limit is each depot's 2021 share of buses applied to the 2025 fleet, plus 10 %: a prior, not a hard limit (brief 4.2).

In [ ]:
HIST_2021 = pd.DataFrame({"depot_name": ["Carabanchel", "Fuencarral", "La Elipa", "Entrevías", "Sanchinarro"],
                          "buses_2021": [458, 486, 314, 399, 438], "lines_2021": [51, 41, 36, 42, 49]})
assume("Buses and lines per depot, 2021", HIST_2021.set_index("depot_name").to_dict("index"), "V",
       "EMT Resumen Ejecutivo 2021, 'Infraestructuras', p. 26 (brief 4.2); historical prior, not a 2025 count", "§5 capacity check")

# Where each line starts and ends: first and last stop of every reference-day trip.
with zipfile.ZipFile(PATHS["emt_gtfs.zip"]) as z:
    stops = pd.read_csv(z.open("stops.txt"), dtype=str, usecols=["stop_id", "stop_lat", "stop_lon"]).astype({"stop_lat": float, "stop_lon": float})
    st5 = pd.read_csv(z.open("stop_times.txt"), dtype=str, usecols=["trip_id", "stop_id", "stop_sequence"])
st5 = (st5[st5.trip_id.isin(t.trip_id)].assign(seq=lambda d: d.stop_sequence.astype(int)).sort_values(["trip_id", "seq"])
       .merge(t[["trip_id", "route_id"]], on="trip_id").merge(stops, on="stop_id"))
ends = pd.concat([st5.groupby("trip_id").head(1), st5.groupby("trip_id").tail(1)]).drop_duplicates(["route_id", "stop_id"])
mids = st5.drop_duplicates(["route_id", "stop_id"]).groupby("route_id")[["stop_lat", "stop_lon"]].mean()

def depot_km(points, by_route=True):
    """Straight-line km from each line to each depot: the closest of its terminals (or its stop centroid)."""
    out = pd.DataFrame({d.depot_name: haversine_km(points.stop_lat, points.stop_lon, d.latitude, d.longitude)
                        for _, d in DEPOTS.iterrows()}, index=points.index)
    return (out.join(points[["route_id"]]).groupby("route_id").min() if by_route else out).reindex(lines.index)

DIST = depot_km(ends)                        # primary: nearest terminal
DIST_MID = depot_km(mids, by_route=False)    # alternative: centre of the line

RESERVE_R = assume("Reserve ratio r: share of the fleet not on the road at the network peak",
                   round(1 - lines.peak_buses.sum() / BASELINE_2025["fleet_total"], 4), "D",
                   f"1 - {int(lines.peak_buses.sum())} peak buses (File 2) / {BASELINE_2025['fleet_total']} buses (§3 baseline); same r at every depot",
                   "§5 fleet per depot (not File 3)", "Depot-specific r from 2021 shares")
CAP_TOL = assume("Depot capacity check: allowed excess over the 2021 size scaled to the 2025 fleet", 0.10, "A",
                 "2021 counts are a prior, not a limit (brief 4.2); 10 % absorbs growth and re-planning since 2021", "§5", "0 % and 20 %")

def capacity(depots=None, tol=CAP_TOL):
    h = HIST_2021.set_index("depot_name").buses_2021
    h = h if depots is None else h[depots]
    return h / h.sum() * BASELINE_2025["fleet_total"] * (1 + tol)

def allocate(dist, cap=None):
    """Nearest depot; then, while a depot's fleet (active / (1 - r)) exceeds its capacity, move the line whose
    next-best depot with room adds the fewest km. Returns a Series route_id -> depot."""
    dep = dist.idxmin(axis=1)
    if cap is None:
        return dep
    need = lines.peak_buses / (1 - RESERVE_R)
    for _ in range(len(dep)):
        fleet = need.groupby(dep).sum().reindex(cap.index, fill_value=0)
        over = fleet - cap
        if (over <= 0).all():
            break
        d0, best = over.idxmax(), None
        for rid in dep.index[dep == d0]:
            for alt in cap.index:
                if alt != d0 and fleet[alt] + need[rid] <= cap[alt]:
                    extra = dist.at[rid, alt] - dist.at[rid, d0]
                    if best is None or extra < best[0]:
                        best = (extra, rid, alt)
        if best is None:
            break
        dep[best[1]] = best[2]
    return dep

CAP = capacity()
DEPOT_OF = allocate(DIST, CAP)
line_table["depot_name"] = DEPOT_OF
lines["depot_name"] = DEPOT_OF

def depot_summary(dep):
    s = pd.DataFrame({"lines": dep.value_counts(), "active_buses": lines.peak_buses.groupby(dep).sum()}).reindex(HIST_2021.depot_name).fillna(0).astype(int)
    raw = s.active_buses / (1 - RESERVE_R)
    fl = np.floor(raw).astype(int)                       # largest-remainder rounding so the total is exactly the baseline fleet
    fl[(raw - fl).sort_values(ascending=False).index[:BASELINE_2025["fleet_total"] - fl.sum()]] += 1
    s["fleet_est"] = fl
    return s

SUMMARY = depot_summary(DEPOT_OF).join(HIST_2021.set_index("depot_name"))
SUMMARY["capacity_check"] = CAP.round(0).astype(int)
SUMMARY["fleet_vs_2021_pct"] = ((SUMMARY.fleet_est / SUMMARY.buses_2021 - 1) * 100).round(1)
SUMMARY.loc["total"] = SUMMARY.sum(numeric_only=True)
SUMMARY.loc["total", "fleet_vs_2021_pct"] = round((SUMMARY.at["total", "fleet_est"] / SUMMARY.at["total", "buses_2021"] - 1) * 100, 1)
display(SUMMARY)

pure = allocate(DIST)
moved = (DEPOT_OF != pure)
km_line = pd.Series({rid: DIST.at[rid, DEPOT_OF[rid]] for rid in DIST.index})
margin = DIST.apply(lambda r: r.nsmallest(2).diff().iloc[-1], axis=1)
print(f"Moved by the capacity check: {moved.sum()} lines, {int(lines.peak_buses[moved].sum())} peak buses "
      f"({list(lines.loc[moved, 'line_id'])[:15]}{'…' if moved.sum() > 15 else ''})")
print(f"Terminal-to-depot distance, assigned depot: median {km_line.median():.1f} km, max {km_line.max():.1f} km (line {lines.line_id[km_line.idxmax()]})")
print(f"Lines whose two nearest depots are within 1 km of each other (either is plausible): {(margin < 1).sum()}")

check("Every line has exactly one primary depot", DEPOT_OF.notna().all() and DEPOT_OF.index.equals(lines.index))
check("buses_assigned identity: active buses by depot add up to File 2 peak buses",
      SUMMARY.loc[HIST_2021.depot_name, "active_buses"].sum() == lines.peak_buses.sum())
check("Fleet per depot reconciles to the §3 baseline", SUMMARY.loc[HIST_2021.depot_name, "fleet_est"].sum() == BASELINE_2025["fleet_total"])
check("No depot above its capacity check after reallocation",
      (SUMMARY.loc[HIST_2021.depot_name, "fleet_est"] <= CAP.round(0) + 1).all())
check("Validation: line 34 at Carabanchel (EMT tender for 120 e-buses, brief 4.2)", DEPOT_OF.get("034") == "Carabanchel",
      f"line 34 -> {DEPOT_OF.get('034')}")

**Alternatives.** The same summary under rules A–C. A depot whose active buses move a lot between rules is one where the allocation, and therefore its readiness in §6, is uncertain. The depots come in close pairs (Fuencarral–Sanchinarro 3 km apart, Carabanchel–Entrevías 4 km), so for many lines the two nearest depots are within 1 km of each other: within a pair the split is weakly determined, while the split between north, east and south is robust. This is also why Sanchinarro ends below its 2021 size: Fuencarral wins most near-ties.

In [ ]:
alts = {
    "Primary: nearest terminal + capacity": DEPOT_OF,
    "A · nearest terminal, no capacity check": pure,
    "B · nearest line centre + capacity": allocate(DIST_MID, CAP),
    "C · La Elipa closed (nearest of four + capacity)": allocate(DIST.drop(columns="La Elipa"), capacity(["Carabanchel", "Fuencarral", "Entrevías", "Sanchinarro"])),
}
cmp_ = pd.DataFrame({k: lines.peak_buses.groupby(v).sum() for k, v in alts.items()}).reindex(HIST_2021.depot_name).fillna(0).astype(int)
cmp_["2021 buses x (1 - r)"] = (HIST_2021.set_index("depot_name").buses_2021 / 2095 * lines.peak_buses.sum()).round(0).astype(int)
print("Active buses per depot under each rule (sum of peak buses):")
display(cmp_)
print("Lines that change depot vs the primary rule:",
      {k: int((v != DEPOT_OF).sum()) for k, v in alts.items() if not k.startswith("Primary")})

# TODO(owner: line model) deadheading sensitivity: DIST (terminal-to-depot km per line) is now available for §4

## 6 · Depot energy, power and readiness (File 3)
`night_energy_mwh = Σ peak_buses × energy_kwh_bus_day / 1000` (all assigned lines); `peak_mw_unmanaged = buses × 100 kW`; `peak_mw_managed = energy ÷ (7 h × 0.90)`. Connection: Sanchinarro 8 MW main + 5 MW backup is **Evidenced**; other depots use a documented proxy (**Estimated**). Readiness per Rule 4, plus at least one alternative connection hypothesis.

> **Decision record · depot grid connections** (open: workstream B decides; options in the team plan)
> | Option | How it works | Main strength | Main weakness |
> |---|---|---|---|
> | H1 · Chargers × simultaneity | Published chargers per depot × kW × share charging at once | Depot-specific, EMT's own evidence | Chargers installed ≠ grid capacity; patchy data |
> | H2 · Nearest grid-node capacity | UFD / e-distribución / i-DE published spare capacity | Official, comparable | Indicative only; node may not serve the depot (never "verified") |
> | H3 · Scale Sanchinarro | 8 MW ÷ 438 buses (2021) ≈ 18 kW per bus × depot size | Anchored on the only evidenced figure | Same readiness for every depot; may flatter the others |
> | H4 · Low / base / high band | A range per depot instead of one value | Shows where readiness flips | Still needs one base value for File 3 |
>
> Indicative pre-screen before allocation: the city needs about 72 MW of managed charging; weighted by 2021 depot sizes, Sanchinarro needs about 15 MW, so **Critical** on 8 MW, **Upgrade** if the 5 MW backup counts.

In [ ]:
def depot_table(file2, depots):
    """depots: depot_name, latitude, longitude, connection_mw_estimate, connection_evidence."""
    g = file2.assign(e=file2.peak_buses * file2.energy_kwh_bus_day / 1000).groupby("depot_name").agg(
        n_lines_assigned=("line_id", "count"), buses_assigned=("peak_buses", "sum"),
        night_energy_mwh=("e", "sum"), conversion_year=("electrification_year", "max")).reset_index()
    d = depots.merge(g, on="depot_name", how="left")
    d["peak_mw_unmanaged"] = d.buses_assigned * P["charger_kw"] / 1000
    d["peak_mw_managed"] = d.night_energy_mwh / (P["window_h"] * P["charge_eff"])
    c = d.connection_mw_estimate
    d["readiness_status"] = np.where(d.peak_mw_managed <= c, "Ready", np.where(d.peak_mw_managed <= 1.5 * c, "Upgrade", "Critical"))
    return d

**Connection capacity.** Sanchinarro is evidenced; the other depots need a documented proxy and at least one alternative hypothesis (Rule 4).

In [ ]:
# TODO(owner: depots) decide the base option (H1-H4) and the alternative hypothesis from the decision record above
# TODO   decide whether Sanchinarro's 5 MW backup counts towards its connection (Evidenced)
# TODO   build connection_mw_estimate per depot, connection_evidence = "Evidenced" (Sanchinarro) or "Estimated"
# TODO   readiness under the base AND the alternative hypothesis; list depots whose status flips (Rule 4 sensitivity)
# TODO   La Elipa 2025 status; Las Tablas only as a future scenario

## 7 · Prioritisation (completes File 2)
Multi-criteria `priority_rank`, then `electrification_year` under S1 so a better rank never has a later year. Use 2025 only for lines already fully zero-emission at the baseline (45 electric lines + line 145 hydrogen, per evidence).

**Why the criteria are transformed and rescaled (ML I study guide).** The criteria come in different units (tonnes of CO₂, riders, a category), so weighting them raw would let the largest unit dominate. Each criterion is:
1. **log-transformed** if right-skewed (CO₂ avoided, riders): shrinks the long tail so a few very large lines do not swamp the rest;
2. **min-max rescaled to 0–1**, so weights read directly as shares of the score;
3. checked with a **correlation filter**: two criteria above |r| = 0.8 carry the same information, keep one.

Standardisation (z-score) is run as an alternative scaling; if both give nearly the same ranking, the choice of scaling does not drive the result.

> **Provisional.** Weights are placeholders until workstream C decides them, and depot readiness joins as a criterion once §6 exists.

In [ ]:
# Provisional scoring with the criteria available today. TODO(owner: prioritisation) confirm criteria and weights.
pr = line_table.copy()
pr["riders_per_day"] = riders_sup.reindex(pr.index)
pr["feasibility"] = pr.charging_strategy.map({"Depot": 1.0, "Opportunity": 0.5, "Not_feasible": 0.0})
CRITERIA = assume("Priority criteria: (log transform?, weight)", {
    "co2_avoided_t_year": (True, 0.40),     # climate impact
    "riders_per_day": (True, 0.30),         # exposure / visibility
    "feasibility": (False, 0.30),           # easiest conversions first
    # TODO(owner: prioritisation) add depot readiness from §6 (Ready = 1, Upgrade = 0.5, Critical = 0) and re-weight
}, "A", "Placeholder weights pending workstream C", "priority_rank, electrification_year", "Scaling and weight sensitivity below")

def criterion_values(df, log):
    """Transformed criteria: log1p where flagged; missing riders filled with the median (logged as an imputation)."""
    out = {}
    for col, (use_log, _) in CRITERIA.items():
        v = df[col].fillna(df[col].median())
        out[col] = np.log1p(v) if use_log else v
    return pd.DataFrame(out)

def score(df, scaling="minmax"):
    v = criterion_values(df, log=True)
    s = (v - v.min()) / (v.max() - v.min()) if scaling == "minmax" else (v - v.mean()) / v.std()
    return sum(w * s[c] for c, (_, w) in CRITERIA.items())

print(f"Riders missing for {pr.riders_per_day.isna().sum()} lines: filled with the median for scoring only")
display(criterion_values(pr, log=True).corr().round(2))
corr = criterion_values(pr, log=True).corr().where(lambda m: ~np.eye(len(m), dtype=bool))
# TODO(owner: prioritisation) CO2 avoided and riders per day correlate at about 0.81 (both grow with line size): keep one,
# TODO   or replace riders per day with a size-free measure such as riders per peak bus, then re-run this filter.
check("Criteria filter: no two criteria correlated above |r| = 0.8", (corr.abs() <= 0.8).all().all(),
      f"max |r| = {corr.abs().max().max():.2f}")

pr["score"] = score(pr, "minmax")
pr["priority_rank"] = pr.score.rank(ascending=False, method="first").astype(int)
alt_rank = score(pr, "zscore").rank(ascending=False, method="first")
rho = pr.priority_rank.corr(alt_rank, method="spearman")
check("Scaling sensitivity: min-max and z-score rankings agree (Spearman >= 0.95)", rho >= 0.95, f"rho = {rho:.3f}")
display(pr.sort_values("priority_rank")[["line_id", "priority_rank", "score", "co2_avoided_t_year", "riders_per_day",
                                         "charging_strategy", "bus_type"]].head(10).round(2))

**Electrification year: equal-count bins (`pd.qcut`).** Lines are cut by priority rank into five groups of equal size, one per year 2026–2030, so the best-ranked fifth converts first and a better rank can never get a later year. Lines already zero-emission at 31 Dec 2025 get 2025.

> **Decision record · year bins** (open: workstream C decides)
> | Option | Pros | Cons |
> |---|---|---|
> | **Equal number of lines per year (`qcut` on rank)** — implemented | Simple, transparent; rank–year consistency is automatic | Buses per year can be uneven (big lines bunch up) |
> | Equal number of buses per year (cumulative peak buses) | Smooth purchases and deliveries, closer to how S1 is bought | Lines per year vary |
> | Depot by depot (a depot converts in one year) | Matches how charging works are done | Needs §5–§6; can conflict with the rank |

In [ ]:
ZE_2025_LINES = []   # TODO(owner: prioritisation) route_ids already fully zero-emission at 31 Dec 2025 (45 electric + line 145 hydrogen), with sources
YEARS = list(range(2026, 2031))
todo = pr.index.difference(ZE_2025_LINES)
pr.loc[ZE_2025_LINES, "electrification_year"] = 2025
pr.loc[todo, "electrification_year"] = pd.qcut(pr.loc[todo, "priority_rank"], q=len(YEARS), labels=YEARS).astype(int)
pr["electrification_year"] = pr.electrification_year.astype(int)

per_year = pr.groupby("electrification_year").agg(lines=("line_id", "count"), peak_buses=("peak_buses", "sum"))
display(per_year.T)
s = pr.sort_values("priority_rank")
check("Better rank never has a later electrification_year", s.electrification_year.is_monotonic_increasing)
print("Alternative (equal buses per year):",
      pd.cut(s.peak_buses.cumsum(), bins=len(YEARS), labels=YEARS).value_counts().sort_index().to_dict(), "lines per year")
# NOTE equal lines per year front-loads buses (about 790 in 2026 vs 150 in 2030) because top-ranked lines are the biggest;
#      equal buses per year spreads purchases evenly, closer to S1 deliveries (about 350 a year). Weigh this in the bin decision.
# TODO(owner: prioritisation) choose the bin rule from the decision record; set ZE_2025_LINES; add depot readiness; then
# TODO   build file2 = pr[F_COLS["File_2.csv"]] with depot_name from §5 and write it in the next cell

**Write Files 2 and 3.** UTF-8, comma-separated, exact column names; the final compliance cell reads them back.

In [ ]:
# file2.to_csv(os.path.join(OUT, "File_2.csv"), index=False, encoding="utf-8")
# file3.to_csv(os.path.join(OUT, "File_3.csv"), index=False, encoding="utf-8")

## 8 · Scenarios 2026–2035 (File 4)
**S0** – 2021–25 pace, adjusted for tenders and deliveries (report both paces). **S1** – 100 % zero-emission by 31 Dec 2030, held to 2035. Explicit yearly purchases and retirements (not inferred from fleet change); service-km allocation by technology; standard/articulated km mix; grid-input energy (÷ 0.90); CNG tonnes; CO₂; avoided CO₂ vs counterfactual (2025 ZE share frozen); capex in constant 2025 €, excl. VAT. Then the earliest credible year and binding constraint, with ≥2 sensitivities.

In [ ]:
# TODO(owner: scenarios)
# purchase/retirement schedules per scenario; fleet mix; energy, fuel, CO2, counterfactual, capex; write File 4


**Earliest credible year.** Compare the binding constraints (purchase pace and budget, depot and grid power, bus deliveries) and run the sensitivities.

In [ ]:
# TODO(owner: scenarios)
# earliest credible year: pace and budget vs depot/grid vs deliveries; sensitivities (e.g. grid decarbonisation, connection proxy, bus life)


## 9 · European benchmark (File 1)
Loads `data_derived/benchmark_gtfs_intermediate.csv` from notebook 02: Madrid plus at least three of the brief's six eligible capitals (London, Paris, Berlin, Brussels, Helsinki, Copenhagen), each on its mandatory network perimeter (brief §4.4), with at least one city fully reproducible from GTFS. Adds Madrid's row from File 2 and the 2025 baseline, plus cited fleet, zero-emission target, charging model and national grid factor per city. Madrid target: 2033 (City of Madrid, Aug 2023).

**Cited fields per city.** The fields File 1 needs that GTFS cannot give: operator, fleet at end 2025, zero-emission fleet, target year, charging model and national grid factor. Each value is labelled with `assume()` and its source; `ze_share_pct` and `energy_full_bev_mwh_day` are computed when File 1 is assembled. Notes and alternatives per city: `notes/section-9-helsinki.md`.

In [1]:
BENCH_CITED = {}

# Helsinki (HSL, whole HSL area). Team decisions, 20 Nov 2026.
BENCH_CITED["Helsinki"] = dict(
    operator="HSL (Helsinki Regional Transport Authority)",
    ze_fleet=assume("Helsinki zero-emission buses, 2025", 600, "V",
                    "HSL 'Electric buses' page, table: 600 electric buses in 2025 (52 %); the page text says about 650",
                    "File 1 Helsinki row", "650 if the page text is used"),
    fleet_total=assume("Helsinki bus fleet, 2025", round(600 / 0.52), "D",
                       "600 electric buses / 52 % share (HSL table); HSL publishes no total. Cross-check: "
                       "934 bus trips on the road at the 08:08 peak on 2026-11-17 (notebook 02), a lower bound",
                       "File 1 Helsinki row", "about 1,250 with the page text (650, 'around half')"),
    fleet_data_year=2025,
    ze_target_year=assume("Helsinki zero-emission target year", 2035, "V",
                          "HSL: public transport in the Helsinki region completely emission-free by 2035",
                          "File 1 Helsinki row", "HSL's own operations: 2030"),
    charging_model=assume("Helsinki charging model", "Depot + opportunity", "V",
                          "HSL: buses charged in depots at night and in daytime breaks; quick charging at some "
                          "terminals or route ends", "File 1 Helsinki row"),
    grid_co2_g_kwh=assume("CO2 factor electricity Finland 2025", 26, "V",
                          "Fingrid: 2025 average for electricity consumed in Finland, 26 gCO2/kWh "
                          "(consumption-based, includes imports; Madrid's factor is generation-based)",
                          "File 1 Helsinki row", "Statistics Finland production-based coefficient (latest 2024)"),
    source_urls=" | ".join([
        "https://www.hsl.fi/en/hsl/electric-buses",
        "https://hsl.fi/en/hsl/responsibility/carbon-neutral-public-transport",
        "https://www.fingrid.fi/en/grid/land-use-and-environment/environmental-impacts-of-transmission-lines/climate-impact/",
        "https://infopalvelut.storage.hsldev.com/gtfs/hsl.zip",
    ]),
)
# TODO(owner: benchmark) add Paris, Berlin, Brussels (and Madrid's operator / target / charging model) in the same shape
_c = BENCH_CITED["Helsinki"]
check("Helsinki: zero-emission fleet does not exceed the fleet", _c["ze_fleet"] <= _c["fleet_total"])
check("Helsinki: derived share matches HSL's published 52 %", abs(_c["ze_fleet"] / _c["fleet_total"] - 0.52) < 0.005,
      f"{100 * _c['ze_fleet'] / _c['fleet_total']:.1f} %")
display(pd.DataFrame(BENCH_CITED).T)

CHECK PASS | Helsinki: zero-emission fleet does not exceed the fleet
CHECK PASS | Helsinki: derived share matches HSL's published 52 % | 52.0 %


,operator,ze_fleet,fleet_total,fleet_data_year,ze_target_year,charging_model,grid_co2_g_kwh,source_urls
Helsinki,HSL (Helsinki Regional Transport Authority),600,1154,2025,2035,Depot + opportunity,26,https://www.hsl.fi/en/hsl/electric-buses | htt...


**File 1 assembly.** Merges notebook 02's GTFS rows (`benchmark_gtfs_intermediate.csv`) with `BENCH_CITED` and Madrid's own row, then writes `File_1.csv`.

In [ ]:
# TODO(owner: benchmark)
# merge intermediate + cited fields; energy_full_bev_mwh_day = daily_bus_km x 1.30 / 1000; write File 1


## 10 · BI visualisation
One self-contained HTML (data embedded, opens by double-click): (i) route map by charging strategy, (ii) depot map with readiness colours and Evidenced/Estimated flag, (iii) benchmark, (iv) S0 vs S1. Each view is also displayed here as a static image (brief 5.3).

**Design.** Colours follow EMT's public website (blue `#2C7ABF`, sky `#128EE1`, charcoal `#414042`, Noto Sans); readiness uses the brief's green / amber / red. Charging strategy uses blue / purple / dashed charcoal so routes never share a colour with depot readiness. A dashed ring marks a depot whose connection is **Estimated**; a solid ring, **Evidenced**. The base map is greyed so route colours carry the meaning. Template and builder: `notebooks/bi/`.

**Data rule.** The HTML embeds Files 1–4 exactly as read back from `outputs/`; route geometry (EMT KML, one direction per line, simplified to about 10 m) is the only other input. A file not yet produced shows as *pending* in its view, never filled with other numbers.

In [ ]:
import sys
sys.path.insert(0, os.path.join(BASE, "notebooks"))
from bi.build_bi import FILES, route_geometry, build_html

BI = {name: (pd.read_csv(os.path.join(OUT, name), encoding="utf-8", dtype={"line_id": str})
             if os.path.exists(os.path.join(OUT, name)) else None) for name in FILES}
routes = route_geometry(os.path.join(RAW, "emt_kml", REF_DATE))
route_to_line = dict(zip(lines.index, lines.line_id))
BI_NOTES = [
    "Views (i) and (ii) are the map; views (iii) and (iv) are the benchmark and scenario tabs (brief 5.3).",
    "Data embedded here are Files 1-4 exactly as saved by notebook 01; a file marked pending has not been produced yet.",
    "Route geometry: EMT KML route maps for the reference day, one direction per line, simplified to about 10 m.",
    "Depot readiness is indicative (Rule 4): managed MW against a connection estimate. A dashed ring means the connection "
    "is a team proxy or scenario, not confirmed grid capacity.",
    f"Charging strategy: Depot within range (300 km standard, 240 km articulated); Opportunity up to {OPP_MAX_RANGE_MULT:g} x range; "
    "Not_feasible beyond (section 4).",
    f"Roadmap: purchases are compared with the unadjusted 2021-25 pace ({S0_PACE_UNADJUSTED:.1f} zero-emission buses a year, section 3); "
    "grid with the sum of File 3 connection estimates; no capex budget reference is set.",
    "Colours follow EMT's public website palette; this is a datathon analysis on EMT open data, not an EMT publication.",
]
BI_PATH = build_html(BI, routes, os.path.join(OUT, "madrid_bus_electrification.html"),
                     dict(subtitle="Line, depot and fleet model · IE Sustainability Datathon", reference_date=REF_DATE, notes=BI_NOTES,
                          pace_ref=S0_PACE_UNADJUSTED, baseline_ze=BASELINE_2025["n_bev"] + BASELINE_2025["n_h2"],
                          footer="Team analysis on EMT Madrid open data (datos.emtmadrid.es). Not an EMT publication. Map © OpenStreetMap contributors."),
                     route_to_line=route_to_line)
print(f"BI file: {BI_PATH} ({os.path.getsize(BI_PATH) / 1e6:.2f} MB) | files embedded: "
      + ", ".join(f"{n} {'yes' if BI[n] is not None else 'pending'}" for n in FILES))
if BI["File_2.csv"] is not None:
    check("BI: every File 2 line has a route on the map", set(BI["File_2.csv"].line_id) <= set(route_to_line.values()))
# TODO(owner: visualisation) static captures of views (i)-(iv) for the jury: display each inline here (e.g. folium/plotly
# TODO   rendering of the same Files 1-4) and export PNGs for the report appendix, with readiness and Evidenced/Estimated legible

## 11 · Decision log and assumptions register
Every material choice was made against at least one alternative; the full records sit in the sections listed. Values are in the assumptions register below.

| Decision | Chosen | Alternatives considered | Status | Section |
|---|---|---|---|---|
| Reference day | Tue 17 Nov 2026 (rule on EMT calendar) | Any Tue/Wed in GTFS validity | Done | §2 |
| Supply day | Tue 9 Jun 2026 (typical within 3 %) | 16 Jun (sensitivity), median of Tuesdays, a Wednesday | Done | §2 |
| 2025 baseline | Fleet register: 2,210 (452 battery, 10 hydrogen, 1,748 CNG), matching the city's official total | EMT accounts mix 456 / 10 / 1,773 (sensitivity); Comunidad 2,215 (noted) | Done | §3 |
| Above-range lines | Opportunity if km per bus-day ≤ 2 × range, else Not_feasible | 1.5 × range (sensitivity); dwell-time rule (no layovers in GTFS); all Not_feasible; all Opportunity | Done | §4 |
| Daily km | KML route × trips (B) | GTFS per trip (A, validation); expanding headways (rejected) | Done | §4 |
| Peak buses for lines without supply | GTFS vehicle blocks | Line-family average, drop the lines (not allowed) | Done | §4 |
| Bus type | Evidence first + proxy (A) | Bracket low/high (B, variance); all Standard; by depot | Done | §4 |
| Outlier treatment | Flag for review, never delete | Delete or cap outliers (rejected: real lines) | Done | §4 |
| University lines' supply day | (c) 9 Jun in File 2, term-time day as sensitivity | (a) 9 Jun only; (b) term-time day in File 2 | Done | §4 |
| Per-bus range | Vehicle blocks (GTFS `block_id`) | Trip span × speed; off-peak buses | Done | §4 |
| Depot allocation | Nearest terminal + capacity check (2021 size scaled to 2025, +10 %); r = 8.0 % | Pure nearest; nearest line centre; La Elipa closed; Las Tablas excluded | Done | §5 |
| Depot connection MW | — | H1 chargers, H2 grid node, H3 scale Sanchinarro, H4 band | Open (B) | §6 |
| Criteria scaling | log + min-max | z-score (sensitivity) | Provisional | §7 |
| Correlated criteria | — | Drop CO₂ or riders (r = 0.81); riders per peak bus instead | Open (C) | §7 |
| Weights | Placeholders | — | Open (C) | §7 |
| Year bins | Equal lines per year (`qcut`) | Equal buses per year; depot by depot | Open (C) | §7 |
| Helsinki benchmark (File 1) | Whole HSL area: 390 lines, 289,727 km/day; 600 ZE of 1,154 buses (52 %); target 2035; Depot + opportunity; 26 g/kWh (Fingrid 2025) | Helsinki city only (183 lines, 131,343 km/day); 650 ZE / ~1,250 buses; Statistics Finland production factor | Done | §9, notebook 02 |

In [ ]:
display(pd.DataFrame(ASSUMPTIONS))

## 12 · Final compliance cell
Reads `File_1.csv`–`File_4.csv` back **as saved** and prints columns, data types, row counts and `head(10)` for each, plus the Rule 3 cross-file checks and the File 4 identities. Mandatory (brief 5.1); a missing printout risks disqualification.

In [ ]:
# FINAL COMPLIANCE CELL (brief 5.1 and Rule 3). Reads the four CSVs back AS SAVED and prints everything.
F_COLS = {
 "File_1.csv": ["city","operator","reference_date","n_bus_lines","daily_bus_km","fleet_total","ze_fleet","ze_share_pct",
                "fleet_data_year","ze_target_year","charging_model","energy_full_bev_mwh_day","grid_co2_g_kwh","source_urls"],
 "File_2.csv": ["line_id","daily_km","peak_buses","km_per_bus_day","bus_type","energy_kwh_bus_day","charging_strategy",
                "depot_name","priority_rank","electrification_year","co2_avoided_t_year"],
 "File_3.csv": ["depot_name","latitude","longitude","n_lines_assigned","buses_assigned","night_energy_mwh","peak_mw_unmanaged",
                "peak_mw_managed","connection_mw_estimate","connection_evidence","readiness_status","conversion_year"],
 "File_4.csv": ["year","scenario","fleet_total","n_bev","n_h2","n_cng","ze_share_pct","electricity_mwh","cng_t","co2_t",
                "co2_avoided_t","capex_meur"],
}
CHECKS.clear()
F = {}
for name, cols in F_COLS.items():
    path = os.path.join(OUT, name)
    print("=" * 100, f"\n{name}")
    if not check(f"{name} exists", os.path.exists(path)):
        continue
    df = pd.read_csv(path, encoding="utf-8", dtype={"line_id": str})
    F[name] = df
    print(f"rows: {len(df)} | columns: {len(df.columns)}")
    print(df.dtypes.to_string())
    display(df.head(10))
    check(f"{name} has exactly the required columns, in order", list(df.columns) == cols,
          "" if list(df.columns) == cols else f"got {list(df.columns)}")

close = lambda a, b, tol=1e-3: abs(float(a) - float(b)) <= tol * max(1.0, abs(float(b)))
print("=" * 100, "\nCross-file and internal checks")
if {"File_2.csv", "File_3.csv"} <= F.keys():
    f2, f3 = F["File_2.csv"], F["File_3.csv"]
    check("File 2: priority_rank unique 1..N", sorted(f2.priority_rank) == list(range(1, len(f2) + 1)))
    s = f2.sort_values("priority_rank")
    check("File 2: better rank never has a later electrification_year", s.electrification_year.is_monotonic_increasing)
    check("File 2: electrification_year within 2025-2030", f2.electrification_year.between(2025, 2030).all())
    rng = f2.bus_type.map(P["range_km"])
    depot_ok = ((f2.km_per_bus_day <= rng) == (f2.charging_strategy == "Depot")).all()
    check("File 2: Depot if and only if km_per_bus_day <= range (Rule 2)", depot_ok)
    check("File 2: energy = km_per_bus_day x kWh/km", np.allclose(f2.energy_kwh_bus_day, f2.km_per_bus_day * f2.bus_type.map(P["kwh_km"]), rtol=1e-3))
    check("All File 2 depot names appear in File 3", set(f2.depot_name) <= set(f3.depot_name))
    agg = f2.assign(e=f2.peak_buses * f2.energy_kwh_bus_day / 1000).groupby("depot_name").agg(
        n=("line_id", "count"), b=("peak_buses", "sum"), e=("e", "sum"), y=("electrification_year", "max"))
    m = f3.set_index("depot_name").join(agg, how="left")
    m = m[m.n.notna()]
    check("File 3 n_lines_assigned reproducible from File 2", (m.n_lines_assigned == m.n).all())
    check("File 3 buses_assigned = sum of File 2 peak_buses", (m.buses_assigned == m.b).all())
    check("File 3 night_energy_mwh reproducible from File 2", np.allclose(m.night_energy_mwh, m.e, rtol=1e-3))
    check("File 3 conversion_year = max electrification_year", (m.conversion_year == m.y).all())
    check("File 3 peak_mw_unmanaged = buses x 100 kW", np.allclose(f3.peak_mw_unmanaged, f3.buses_assigned * P["charger_kw"] / 1000, rtol=1e-3))
    check("File 3 peak_mw_managed = energy / (7 x 0.90)", np.allclose(f3.peak_mw_managed, f3.night_energy_mwh / (P["window_h"] * P["charge_eff"]), rtol=1e-3))
    rd = np.where(f3.peak_mw_managed <= f3.connection_mw_estimate, "Ready",
                  np.where(f3.peak_mw_managed <= 1.5 * f3.connection_mw_estimate, "Upgrade", "Critical"))
    check("File 3 readiness_status follows Rule 4", (rd == f3.readiness_status).all())
    check("File 3 connection_evidence in {Evidenced, Estimated}", f3.connection_evidence.isin(["Evidenced", "Estimated"]).all())
    check("File 3 depot names are EMT operations centres", set(f3.depot_name) <= {"Carabanchel", "Fuencarral", "La Elipa", "Entrevías", "Sanchinarro", "Las Tablas"})
    check("File 2 bus_type in {Standard, Articulated}", f2.bus_type.isin(["Standard", "Articulated"]).all())
    check("File 2 charging_strategy in {Depot, Opportunity, Not_feasible}", f2.charging_strategy.isin(["Depot", "Opportunity", "Not_feasible"]).all())
    check("File 2 line_id unique", f2.line_id.is_unique)
    if "ref_lines" in globals():
        check("File 2: one row per line active on the reference day", len(f2) == len(ref_lines), f"{len(f2)} rows vs {len(ref_lines)} lines in GTFS")
    co2 = f2.daily_km * P["service_days"] * (f2.bus_type.map(P["cng_kg_km"]) * P["cng_kgco2_per_kg"]
          - f2.bus_type.map(P["kwh_km"]) / P["charge_eff"] * P["grid_gco2_kwh"] / 1000) / 1000
    check("File 2 co2_avoided_t_year reproducible from the fixed factors", np.allclose(f2.co2_avoided_t_year, co2, rtol=1e-3))
if {"File_1.csv", "File_2.csv"} <= F.keys():
    f1, f2 = F["File_1.csv"], F["File_2.csv"]
    mad = f1[f1.city == "Madrid"]
    if check("File 1 has a Madrid row", len(mad) == 1):
        mad = mad.iloc[0]
        check("File 1 has Madrid + >= 3 eligible capitals", len(f1) >= 4 and set(f1.city) - {"Madrid"} <= {"London","Paris","Berlin","Brussels","Helsinki","Copenhagen"})
        check("Madrid daily_bus_km = sum(File 2 daily_km)", close(mad.daily_bus_km, f2.daily_km.sum()))
        check("Madrid n_bus_lines = count(File 2 rows)", int(mad.n_bus_lines) == len(f2))
        check("Madrid reference_date = File 2 reference date", str(mad.reference_date) == REF_DATE, f"{mad.reference_date} vs {REF_DATE}")
        check("Madrid fleet_total = 2025 baseline", int(mad.fleet_total) == BASELINE_2025["fleet_total"])
        check("Madrid ze_fleet = 2025 baseline", int(mad.ze_fleet) == BASELINE_2025["n_bev"] + BASELINE_2025["n_h2"])
    check("File 1 ze_share_pct = ze_fleet / fleet_total x 100", np.allclose(f1.ze_share_pct, (f1.ze_fleet / f1.fleet_total * 100).round(1), atol=0.051))
    k = f1.daily_bus_km.notna()
    check("File 1 energy_full_bev_mwh_day = daily_bus_km x 1.30 / 1000", np.allclose(f1.energy_full_bev_mwh_day[k], f1.daily_bus_km[k] * P["benchmark_kwh_km"] / 1000, rtol=1e-3))
    check("File 1 source_urls populated for every row", f1.source_urls.fillna("").str.len().gt(0).all())
if "File_4.csv" in F:
    f4 = F["File_4.csv"]
    check("File 4 has exactly 20 rows (2026-2035 x S0, S1)", len(f4) == 20 and set(zip(f4.year, f4.scenario)) == {(y, s) for y in range(2026, 2036) for s in (0, 1)})
    check("File 4: n_bev + n_h2 + n_cng = fleet_total (every row)", (f4.n_bev + f4.n_h2 + f4.n_cng == f4.fleet_total).all())
    check("File 4: ze_share_pct = 100 x (n_bev + n_h2) / fleet_total", np.allclose(f4.ze_share_pct, (100 * (f4.n_bev + f4.n_h2) / f4.fleet_total).round(1), atol=0.051))
    s1 = f4[(f4.scenario == 1) & (f4.year >= 2030)]
    check("File 4: S1 ze_share_pct = 100 from 2030 to 2035", (s1.ze_share_pct == 100).all())
    check("File 4: S1 n_cng = 0 from 2030 to 2035", (s1.n_cng == 0).all())
print("\nBaseline 31 Dec 2025 used by Files 1 and 4:", BASELINE_2025 if "BASELINE_2025" in globals() else "NOT SET")
summary = pd.DataFrame(CHECKS)
display(summary)
print(f"\n{(summary.result == 'PASS').sum()} PASS / {(summary.result == 'FAIL').sum()} FAIL")